# Recommender Systems — 5‑Day Lab Notebook

This notebook is the **main lab material** for our 5‑day block on recommender systems.

We will go from:

- basic *content‑based* recommenders  
- to *personalized* recommenders  
- to *hybrid & constraint‑aware* recommenders  
- to *agentic* recommenders with a simple **critic agent**  
- plus **visualisation**, **interactive filtering**, and **evaluation**

All code is designed to run on **Google Colab**.

---

## Application domains (choose one per group)

In this lab, each group focuses on **one domain**:

1. **Fashion** – personal stylist for clothing items (e‑commerce)  
2. **Recipes** – everyday cooking companion  
3. **Scholar** – paper navigator for research topics  
4. **Movies** – movie night recommender + offline evaluation

We will use the same **pipeline pattern** for all four domains.  
You select the domain via the `PROJECT` variable below.


---
## 0. Global configuration

- `PROJECT` – select one domain per group.  
- `EMBEDDING_MODEL` – you can later play with different models.  
- `MAX_ROWS` – how many items to load (large datasets can be subsampled for speed).


In [1]:
# =========================
# GLOBAL CONFIGURATION
# =========================

# Options: "fashion", "recipes", "scholar", "movies", "music"
PROJECT = "music"      # 👈 change this per group

# Text embedding model from sentence-transformers.
# You can later try: "all-mpnet-base-v2", "multi-qa-MiniLM-L6-cos-v1", etc.
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# Control dataset size. If None -> full data; if int -> random subset.
MAX_ROWS = 5000

print("Selected PROJECT:", PROJECT)
print("Embedding model :", EMBEDDING_MODEL)
print("MAX_ROWS        :", MAX_ROWS)


Selected PROJECT: music
Embedding model : all-MiniLM-L6-v2
MAX_ROWS        : 5000


---
## 1. Environment setup

We install the libraries we need:

- `sentence-transformers` for text embeddings  
- `ipywidgets` for sliders / interactive filters  
- `gradio` for an optional simple web UI (user‑study sketch)


In [3]:
!pip install sentence-transformers ipywidgets gradio --quiet
import pandas as pd
import numpy as np
import random

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from ipywidgets import interact, widgets
from IPython.display import display, HTML

pd.set_option("display.max_colwidth", 160)



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
def first_existing(df: pd.DataFrame, candidates):
    """Return the first column from 'candidates' that exists in df."""
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"None of {candidates} found in columns: {df.columns.tolist()}")


def display_gallery(recs: pd.DataFrame,
                    title: str = "Recommendations",
                    text_cols=None):
    """Simple HTML gallery view.

    If an image URL column happens to exist ('image_url', 'img', 'main_image'),
    it will be displayed. Otherwise we show text cards.

    This is intentionally generic and works for all four projects.
    """
    if recs is None or len(recs) == 0:
        display(HTML("<p><i>No items to display.</i></p>"))
        return

    if text_cols is None:
        if PROJECT == "fashion":
            text_cols = ["product_name", "color", "category_tree", "final_price"]
        elif PROJECT == "recipes":
            text_cols = ["title", "ingredients"]
        elif PROJECT == "scholar":
            text_cols = ["title", "abstract"]
        elif PROJECT == "movies":
            text_cols = ["title", "genres"]
        else:
            text_cols = list(recs.columns)

    # try to find an image column if there is one (many datasets won't have it)
    img_col = None
    for c in ["image_url", "img", "main_image"]:
        if c in recs.columns:
            img_col = c
            break

    cards_html = []
    for _, row in recs.iterrows():
        # build text part
        parts = []
        for col in text_cols:
            if col in recs.columns:
                val = str(row[col])
                parts.append(f"<div><b>{col}</b>: {val}</div>")

        # build image part if available
        img_html = ""
        if img_col is not None and isinstance(row.get(img_col), str):
            url = row[img_col]
            if url and url.startswith("http"):
                img_html = f"""
                <div style='height:150px; overflow:hidden; margin-bottom:6px;'>
                    <img src='{url}' style='width:100%; object-fit:cover;'>
                </div>
                """

        card = f"""
        <div style='width:230px; margin:8px; padding:10px; border-radius:12px;
                    box-shadow:0 2px 6px rgba(0,0,0,0.1); background:white; font-size:12px;'>
            {img_html}
            {''.join(parts)}
        </div>
        """
        cards_html.append(card)

    html = f"""
    <h3>{title}</h3>
    <div style='display:flex; flex-wrap:wrap; gap:6px; background:#f5f5f5; padding:10px; border-radius:12px;'>
        {''.join(cards_html)}
    </div>
    """
    display(HTML(html))


---
# 📅 Day 1 — Data, Item Representations & Content-Based Recommendation

### Goals

- Load a **real dataset** for your chosen domain.  
- Turn items into **vector representations** (embeddings).  
- Compute **content-based similarity** between items.  
- Visualise the embedding space in **2D and 3D**.


## 1.1 Load a real dataset

We work with four public datasets:

- **Fashion**: sample of H&M product catalogue (titles, descriptions, color, category, price).  
- **Recipes**: ~13k recipes (title, ingredients, instructions).  
- **Scholar**: AI-generated vs human-like abstracts (we treat them as "papers").  
- **Movies**: MovieLens `ml-latest-small` movies + ratings.

We always return:

- `df` — item table (one row per item)  
- `ratings` — optional rating table (only for movies; used later for offline evaluation)


In [4]:
FASHION_URL = "https://raw.githubusercontent.com/luminati-io/HM-dataset-sample/main/H%26M-dataset-sample.csv"
RECIPES_URL = "https://raw.githubusercontent.com/josephrmartinez/recipe-dataset/main/13k-recipes.csv"
SCHOLAR_URL = "https://raw.githubusercontent.com/panagiotisanagnostou/AI-GA/main/ai-ga-dataset.csv"
MOVIES_URL  = "https://raw.githubusercontent.com/maharjansudhan/DATA612/master/ml-latest-small/movies.csv"
RATINGS_URL = "https://raw.githubusercontent.com/maharjansudhan/DATA612/master/ml-latest-small/ratings.csv"
MUSIC_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"

def load_project_data(project: str):
    ratings = None

    if project == "fashion":
        df = pd.read_csv(FASHION_URL)
        # keep core columns
        keep = [c for c in ["product_name", "description", "color", "category_tree", "final_price"] if c in df.columns]
        df = df[keep].dropna(subset=[keep[0]]).reset_index(drop=True)
        df["id"] = df.index.astype(str)
        # Clean and convert final_price to numeric
        if "final_price" in df.columns:
            df["final_price"] = df["final_price"].astype(str).str.replace('"', '').str.replace('$', '', regex=False)
            df["final_price"] = pd.to_numeric(df["final_price"], errors='coerce')

    elif project == "recipes":
        df = pd.read_csv(RECIPES_URL)
        title_col = first_existing(df, ["Title", "title"])
        ing_col   = first_existing(df, ["Ingredients", "ingredients"])
        instr_col = first_existing(df, ["Instructions", "instructions", "Directions"])
        df = df[[title_col, ing_col, instr_col]].rename(columns={title_col: "title",
                                                                 ing_col: "ingredients",
                                                                 instr_col: "instructions"})
        df = df.dropna().reset_index(drop=True)
        df["id"] = df.index.astype(str)

    elif project == "scholar":
        df = pd.read_csv(SCHOLAR_URL)
        title_col    = first_existing(df, ["title", "Title"])
        abstract_col = first_existing(df, ["abstract", "Abstract", "text"])
        label_col    = first_existing(df, ["label", "Label"])
        df = df[[title_col, abstract_col, label_col]].rename(columns={title_col: "title",
                                                                      abstract_col: "abstract",
                                                                      label_col: "label"})
        df = df.dropna().reset_index(drop=True)
        df["id"] = df.index.astype(str)

    elif project == "movies":
        movies  = pd.read_csv(MOVIES_URL)
        ratings = pd.read_csv(RATINGS_URL)
        movies["id"] = movies["movieId"].astype(str)
        df = movies[["id", "title", "genres"]].dropna().reset_index(drop=True)

    elif project == "music":
        df  = pd.read_csv(MUSIC_URL)
        df = df.dropna(subset=['track_name', 'track_artist']).reset_index(drop=True)
        df["decade"] = df["track_album_release_date"].apply(lambda x: int(str(x)[:4]) // 10 * 10)
        df["id"] = df["track_id"].astype(str)

    else:
        raise ValueError("PROJECT must be one of: fashion, recipes, scholar, movies, music")

    # Optional sampling for speed
    if MAX_ROWS is not None and len(df) > MAX_ROWS:
        df = df.sample(MAX_ROWS, random_state=42).reset_index(drop=True)

    return df, ratings


df, ratings = load_project_data(PROJECT)
print(f"Loaded {len(df)} items for project='{PROJECT}'. Columns: {df.columns.tolist()}")
df.head()

Loaded 5000 items for project='music'. Columns: ['track_id', 'track_name', 'track_artist', 'track_popularity', 'track_album_id', 'track_album_name', 'track_album_release_date', 'playlist_name', 'playlist_id', 'playlist_genre', 'playlist_subgenre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'decade', 'id']


,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,decade,id
0,3m0BCyPaS5BN5iQgtdPr0u,Perfect (feat. Haris) - LUM!X Remix,Lucas & Steve,59,45DODwRRHNzuqPq18lGAWl,Perfect (feat. Haris) [LUM!X Remix],2019-11-29,Big Room EDM - by Spinnin' Records,7xWdFCrU5Gka6qp1ODrSdK,edm,...,1,0.0354,0.00266,0.000003,0.0581,0.509,127.932,187074,2010,3m0BCyPaS5BN5iQgtdPr0u
1,0u6CqsKqlArqFuMVKREXp9,Never Can Say Goodbye,The Communards,3,5JYaoYQ7d8twJ2gp6kPt6C,Red,1987,80's Songs | Top 💯 80s Music Hits,65HtIbyFkaQPflCa4oW8KO,pop,...,1,0.0287,0.04340,0.027300,0.2860,0.836,132.172,285986,1980,0u6CqsKqlArqFuMVKREXp9
2,505hHgfw2U0XtOpC3XYHOQ,A Waiting Game,Claptone,34,3oGTPoEgDlNxSDMmaWBDS9,Fantast,2018-06-08,Music&Other Drugs,5jROYSZSL7cO0jGAqkPx7C,pop,...,1,0.0465,0.09850,0.000405,0.2380,0.201,143.000,204293,2010,505hHgfw2U0XtOpC3XYHOQ
3,0HVj61S4pQJdQhgZP61G65,I Would Like,Zara Larsson,8,1ChPmbFVHtAYAEIbdDB4a4,I Would Like,2016,ELECTROPOP,2UsEj2GUukV0GLbsE3rldz,pop,...,0,0.0524,0.08530,0.000000,0.0839,0.297,121.028,226720,2010,0HVj61S4pQJdQhgZP61G65
4,6cW7FhUH5KInJVmIfWcOXU,Hawaiian Haze,goosetaf,41,12kYlTM2a3e5ONRZD5XsUl,"Textures, Vol. 1",2018-05-19,Sunny Beats,37i9dQZF1DXbtuVQL4zoey,latin,...,1,0.2220,0.41000,0.538000,0.0451,0.843,79.014,109367,2010,6cW7FhUH5KInJVmIfWcOXU


In [13]:
# Analyze what gets dropped by dropna()
df_original = pd.read_csv(MUSIC_URL)

# Step 1: dropna effect (only on track_name and track_artist)
df_after_dropna = df_original.dropna(subset=['track_name', 'track_artist']).reset_index(drop=True)
rows_before = len(df_original)
rows_after_dropna = len(df_after_dropna)
rows_dropped_by_dropna = rows_before - rows_after_dropna

print("=" * 70)
print("ANALYSIS: df.dropna(subset=['track_name', 'track_artist'])")
print("=" * 70)
print(f"\n1️⃣  DROPNA OPERATION ONLY:")
print(f"   Rows before dropna:        {rows_before:,}")
print(f"   Rows after dropna:         {rows_after_dropna:,}")
print(f"   Rows removed by dropna:    {rows_dropped_by_dropna:,}")
print(f"   Percentage removed:        {(rows_dropped_by_dropna / rows_before * 100):.4f}%")
print()

# Missing value details
print(f"2️⃣  MISSING VALUES BREAKDOWN:")
missing_name = df_original['track_name'].isna().sum()
missing_artist = df_original['track_artist'].isna().sum()
missing_both = (df_original['track_name'].isna() & df_original['track_artist'].isna()).sum()
print(f"   Missing 'track_name':      {missing_name:,}")
print(f"   Missing 'track_artist':    {missing_artist:,}")
print(f"   Missing BOTH:              {missing_both:,}")
print()

# Show sample of dropped rows
print(f"3️⃣  SAMPLE OF DROPPED ROWS:")
dropped_mask = df_original['track_name'].isna() | df_original['track_artist'].isna()
dropped_indices = df_original.index[dropped_mask]
if len(dropped_indices) > 0:
    sample_cols = ['track_name', 'track_artist', 'track_album_name', 'track_id']
    sample_dropped = df_original.loc[dropped_indices, sample_cols].copy()
    print(sample_dropped.to_string())
else:
    print("   (No rows dropped)")
print()

# Note about MAX_ROWS
print(f"4️⃣  NOTE ON MAX_ROWS SAMPLING:")
print(f"   MAX_ROWS is set to:        {MAX_ROWS}")
print(f"   After MAX_ROWS sampling:   {len(df):,} rows")
print(f"   (This is a separate operation that randomly samples the data for speed)")

print("\n" + "=" * 70)

ANALYSIS: df.dropna(subset=['track_name', 'track_artist'])

1️⃣  DROPNA OPERATION ONLY:
   Rows before dropna:        32,833
   Rows after dropna:         32,828
   Rows removed by dropna:    5
   Percentage removed:        0.0152%

2️⃣  MISSING VALUES BREAKDOWN:
   Missing 'track_name':      5
   Missing 'track_artist':    5
   Missing BOTH:              5

3️⃣  SAMPLE OF DROPPED ROWS:
      track_name track_artist track_album_name                track_id
8151         NaN          NaN              NaN  69gRFGOWY9OMpFJgFol1u0
9282         NaN          NaN              NaN  5cjecvX0CmC9gK0Laf5EMQ
9283         NaN          NaN              NaN  5TTzhRSWQS4Yu8xTgAuq6D
19568        NaN          NaN              NaN  3VKFip3OdAvv4OfNTgFWeQ
19811        NaN          NaN              NaN  69gRFGOWY9OMpFJgFol1u0

4️⃣  NOTE ON MAX_ROWS SAMPLING:
   MAX_ROWS is set to:        5000
   After MAX_ROWS sampling:   5,000 rows
   (This is a separate operation that randomly samples the data for speed)


### 1.2 Quick data sanity check

Always inspect a few rows and some basic statistics before building recommenders.


In [14]:
print("Shape:", df.shape)
display(df.head(5))
display(df.describe(include='all').transpose())

Shape: (5000, 25)


,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,decade,id
0,3m0BCyPaS5BN5iQgtdPr0u,Perfect (feat. Haris) - LUM!X Remix,Lucas & Steve,59,45DODwRRHNzuqPq18lGAWl,Perfect (feat. Haris) [LUM!X Remix],2019-11-29,Big Room EDM - by Spinnin' Records,7xWdFCrU5Gka6qp1ODrSdK,edm,...,1,0.0354,0.00266,0.000003,0.0581,0.509,127.932,187074,2010,3m0BCyPaS5BN5iQgtdPr0u
1,0u6CqsKqlArqFuMVKREXp9,Never Can Say Goodbye,The Communards,3,5JYaoYQ7d8twJ2gp6kPt6C,Red,1987,80's Songs | Top 💯 80s Music Hits,65HtIbyFkaQPflCa4oW8KO,pop,...,1,0.0287,0.04340,0.027300,0.2860,0.836,132.172,285986,1980,0u6CqsKqlArqFuMVKREXp9
2,505hHgfw2U0XtOpC3XYHOQ,A Waiting Game,Claptone,34,3oGTPoEgDlNxSDMmaWBDS9,Fantast,2018-06-08,Music&Other Drugs,5jROYSZSL7cO0jGAqkPx7C,pop,...,1,0.0465,0.09850,0.000405,0.2380,0.201,143.000,204293,2010,505hHgfw2U0XtOpC3XYHOQ
3,0HVj61S4pQJdQhgZP61G65,I Would Like,Zara Larsson,8,1ChPmbFVHtAYAEIbdDB4a4,I Would Like,2016,ELECTROPOP,2UsEj2GUukV0GLbsE3rldz,pop,...,0,0.0524,0.08530,0.000000,0.0839,0.297,121.028,226720,2010,0HVj61S4pQJdQhgZP61G65
4,6cW7FhUH5KInJVmIfWcOXU,Hawaiian Haze,goosetaf,41,12kYlTM2a3e5ONRZD5XsUl,"Textures, Vol. 1",2018-05-19,Sunny Beats,37i9dQZF1DXbtuVQL4zoey,latin,...,1,0.2220,0.41000,0.538000,0.0451,0.843,79.014,109367,2010,6cW7FhUH5KInJVmIfWcOXU


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
track_id,5000,4838,0U6bQIAh6MCGo1xjbIIx2S,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track_name,5000,4571,Down,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track_artist,5000,3136,Martin Garrix,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track_popularity,5000.0,NaN,NaN,NaN,42.5666,24.626824,0.0,25.0,45.0,62.0,98.0
track_album_id,5000,4518,3tQd5mwBtVyxCoEo4htGAV,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track_album_name,5000,4304,Greatest Hits,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
track_album_release_date,5000,1868,2019-11-22,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
playlist_name,5000,442,Indie Poptimism,57,NaN,NaN,NaN,NaN,NaN,NaN,NaN
playlist_id,5000,464,37i9dQZF1DWTHM4kX49UKs,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
playlist_genre,5000,6,edm,936,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Calculate complexity for the entire DataFrame if not already present
# if 'complexity' not in df.columns:
#     df['complexity'] = df['ingredients'].apply(lambda x: len(str(x)))

# # Define 'high complexity' as above the 75th percentile
# high_complexity_threshold = df['complexity'].quantile(0.75)
# high_complexity_recipes = df[df['complexity'] >= high_complexity_threshold]

# print(f"Number of high complexity recipes: {len(high_complexity_recipes)}")
# print(f"Complexity threshold (75th percentile): {high_complexity_threshold:.0f}")

# # Plot the distribution
# plt.figure(figsize=(10, 6))
# sns.histplot(high_complexity_recipes['complexity'], bins=30, kde=True)
# plt.title('Distribution of Complexity for High Complexity Recipes')
# plt.xlabel('Complexity (Length of Ingredients String)')
# plt.ylabel('Number of Recipes')
# plt.grid(axis='y', alpha=0.75)
# plt.show()

In [ ]:
# #FOR MOVIE PROJECT

# # ============================================
# # 1. INSTALL PACKAGES (run once)
# # ============================================
# # !pip install pandas requests beautifulsoup4 lxml tqdm


# # ============================================
# # 2. IMPORTS
# # ============================================
# import pandas as pd
# import requests
# import re
# from bs4 import BeautifulSoup
# from time import sleep
# from tqdm import tqdm


# # ============================================
# # 3. LOAD MOVIELENS DATASET
# # ============================================
# url = "https://raw.githubusercontent.com/maharjansudhan/DATA612/master/ml-latest-small/movies.csv"
# df = pd.read_csv(url)

# # Clean movie titles (remove year)
# df["clean_title"] = df["title"].apply(lambda x: re.sub(r"\s\(\d{4}\)", "", x))

# print("Dataset loaded:", df.shape)


# # ============================================
# # 4. WIKIPEDIA SEARCH + RUNTIME SCRAPER (STABLE)
# # ============================================
# headers = {
#     "User-Agent": "Mozilla/5.0"
# }

# def get_wiki_runtime(title):
#     try:
#         # --- Step 1: Use Wikipedia search API ---
#         search_url = "https://en.wikipedia.org/w/api.php"
#         params = {
#             "action": "query",
#             "list": "search",
#             "srsearch": f"{title} film",
#             "format": "json"
#         }

#         r = requests.get(search_url, params=params, headers=headers, timeout=10)
#         data = r.json()

#         if len(data["query"]["search"]) == 0:
#             return None

#         page_title = data["query"]["search"][0]["title"].replace(" ", "_")
#         wiki_url = f"https://en.wikipedia.org/wiki/{page_title}"

#         # --- Step 2: Open movie page ---
#         movie_page = requests.get(wiki_url, headers=headers, timeout=10)
#         soup = BeautifulSoup(movie_page.text, "lxml")

#         # --- Step 3: Find runtime in infobox ---
#         infobox = soup.find("table", class_="infobox")
#         if not infobox:
#             return None

#         rows = infobox.find_all("tr")
#         for row in rows:
#             header = row.find("th")
#             if header and "Running time" in header.text:
#                 time_text = row.find("td").text.strip()

#                 # Extract minutes safely
#                 mins = re.findall(r"\d+", time_text)
#                 if mins:
#                     return int(mins[0])

#         return None

#     except:
#         return None


# # ============================================
# # 5. RUN SCRAPER (FIRST 10 MOVIES - SAFE)
# # ============================================
# df["duration_min"] = None

# N = 10  # ✅ SAFE TEST SIZE

# for i, title in enumerate(tqdm(df["clean_title"].iloc[:N])):
#     runtime = get_wiki_runtime(title)
#     df.loc[i, "duration_min"] = runtime
#     sleep(1.5)   # Safe delay


# # ============================================
# # 6. SAVE RESULT
# # ============================================
# df.to_csv("movies_with_duration.csv", index=False)

# print("✅ Saved as movies_with_duration.csv")
# print(df.head(10)[["movieId", "title", "duration_min"]])


## 1.3 Item embeddings

We convert text fields into dense vectors using a pre‑trained `SentenceTransformer` model:

- Fashion: product name + description + category  
- Recipes: title + ingredients  
- Scholar: title + abstract  
- Movies: title + genres
- Music: title + artist + genre + subgenre + decade + valence + energy


In [6]:
import pickle
import os
import sentence_transformers

# Create folders for embeddings (if not exists)
os.makedirs('embeddings', exist_ok=True)

# Define embedding models to test
EMBEDDING_MODELS = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "multi-qa-MiniLM-L6-cos-v1",
    "mixedbread-ai/mxbai-embed-large-v1",
]

# other models
    # "Snowflake/snowflake-arctic-embed-l-v2.0",
    # "Alibaba-NLP/gte-large-en-v1.5",
    # "thenlper/gte-large",
    # "colbert-ir/colbertv2.0",
    # "naver/splade-cocondenser-ensembledistil",
    # "BAAI/bge-m3",
    # "jinaai/jina-embeddings-v4",
    # "intfloat/e5-mistral-7b-instruct",
    # "nomic-ai/modernbert-embed-base"

# Define save paths
save_folder = 'embeddings'
results = {}
processed_models = []

for model_name in EMBEDDING_MODELS:
    save_path = f'{save_folder}/{model_name.replace("/", "-")}.pkl'

    # Check if embeddings already exist
    if os.path.exists(save_path):
        print(f"\n{'='*50}")
        print(f"Model: {model_name}")
        print('='*50)
        print(f"✅ Embeddings already exist for: {model_name}")
        print(f"   Loaded from: {save_path}")

        # Load existing embeddings
        with open(save_path, 'rb') as f:
            result = pickle.load(f)

        results[model_name] = result
        processed_models.append(model_name)
        continue

    print(f"\n{'='*50}")
    print(f"Testing model: {model_name}")
    print('='*50)
    print("⏳ Processing embeddings (new model)...")

    # Load the text embedding model for this iteration
    text_model = sentence_transformers.SentenceTransformer(model_name)

    # Build texts based on project type
    if PROJECT == "fashion":
        texts = (df["product_name"].fillna("") + " " +
                 df["description"].fillna("") + " " +
                 df.get("category_tree", "").fillna("")).tolist()

    elif PROJECT == "recipes":
        texts = (df["title"].fillna("") + " " +
                df["ingredients"].fillna("")).tolist()

    elif PROJECT == "scholar":
        texts = (df["title"].fillna("") + " " +
                df["abstract"].fillna("")).tolist()

    elif PROJECT == "music":
        texts = (df["track_name"].fillna("") + " " +
                df["track_artist"].fillna("") + " " +
                df["playlist_genre"].fillna("") + " " +
                df["playlist_subgenre"].fillna("") + " " +
                df["valence"].astype(str).fillna("") + " " +
                df["energy"].astype(str).fillna("") + " " +
                df["decade"].astype(str).fillna("")).tolist()

    else:  # movies
        texts = (df["title"].fillna("") + " " +
                df["genres"].fillna("")).tolist()

    print(f"Encoding {len(texts)} items with model: {model_name}")

    # Generate embeddings
    embeddings = text_model.encode(texts, show_progress_bar=True)
    print(f"Embeddings shape: {embeddings.shape}")

    # Store results
    results[model_name] = {
        "embeddings": embeddings,
        "shape": embeddings.shape
    }

    # Save embeddings to local folder
    with open(save_path, 'wb') as f:
        pickle.dump(results[model_name], f)

    print(f"💾 Saved {model_name} embeddings to local folder")


# Summary report
print("\n" + "="*50)
print("PROCESSING SUMMARY")
print("="*50)

already_loaded = len(processed_models)
new_processed = len(results) - already_loaded
total_models = len(results)

print(f"\n✅ Models loaded from disk (already exist): {already_loaded}")
print(f"🔄 Models newly processed: {new_processed}")
print(f"📊 Total models processed: {total_models}")
print(f"\nAvailable model names in results: {list(results.keys())}")
print("="*50)


Model: all-MiniLM-L6-v2
✅ Embeddings already exist for: all-MiniLM-L6-v2
   Loaded from: embeddings/all-MiniLM-L6-v2.pkl

Model: all-mpnet-base-v2
✅ Embeddings already exist for: all-mpnet-base-v2
   Loaded from: embeddings/all-mpnet-base-v2.pkl

Model: multi-qa-MiniLM-L6-cos-v1
✅ Embeddings already exist for: multi-qa-MiniLM-L6-cos-v1
   Loaded from: embeddings/multi-qa-MiniLM-L6-cos-v1.pkl

Model: mixedbread-ai/mxbai-embed-large-v1
✅ Embeddings already exist for: mixedbread-ai/mxbai-embed-large-v1
   Loaded from: embeddings/mixedbread-ai-mxbai-embed-large-v1.pkl

PROCESSING SUMMARY

✅ Models loaded from disk (already exist): 4
🔄 Models newly processed: 0
📊 Total models processed: 4

Available model names in results: ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', 'multi-qa-MiniLM-L6-cos-v1', 'mixedbread-ai/mxbai-embed-large-v1']


## 1.4 Visualising the embedding space (2D & 3D PCA)

We reduce the high‑dimensional embeddings with PCA for exploration.

> This is only for **intuition** — we are NOT training on the 2D/3D projections.


In [18]:
import pickle
import os
import sentence_transformers
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd

# ============================
# ✅ GENERATE PCA VISUALIZATIONS (6 GRAPHS)
# ============================

# -------------------------
# 1. CHOOSE COLOR HERE
# -------------------------
COLOR_BY = 'valence'  # 👈 Changed to 'valence'

# -------------------------
# 2. SAMPLE EMBEDDINGS
# -------------------------
max_points = min(1500, len(df))
np.random.seed(1234)  # Set seed for reproducibility
idx_sample = np.random.choice(len(df), max_points, replace=False)
df_sample = df.iloc[idx_sample].reset_index(drop=True)

# -------------------------
# 3. SAFE HOVER LABEL
# -------------------------
if PROJECT == "fashion" and "product_name" in df_sample.columns:
    hover_col = "product_name"
elif "title" in df_sample.columns:
    hover_col = "title"
elif PROJECT == "music":
    music_hover_cols = [col for col in ["track_name", "track_artist", "valence", "energy", "playlist_subgenre", "decade"] if col in df_sample.columns]
    if music_hover_cols:
        hover_col = music_hover_cols
    else:
        hover_col = df_sample.columns[0]
else:
    hover_col = df_sample.columns[0]

# -------------------------
# 4. DERIVED COLOR FEATURES (SAFE)
# -------------------------
if PROJECT == "recipes" and "ingredients" in df_sample.columns:
    df_sample["complexity"] = df_sample["ingredients"].apply(lambda x: len(str(x)))

if PROJECT == "movies" and "genres" in df_sample.columns:
    df_sample["main_genre"] = df_sample["genres"].astype(str).apply(lambda x: x.split("|")[0])

if PROJECT == "music" and "genres" in df_sample.columns:
    df_sample["main_genre"] = df_sample["genres"].astype(str).apply(lambda x: x.split("|")[0])

# -------------------------
# 5. SELECT TWO RANDOM SONGS TO HIGHLIGHT
# -------------------------
np.random.seed(1234)
highlight_indices = np.random.choice(len(df_sample), 2, replace=False)
highlight_indices = sorted(highlight_indices.tolist())

df_sample["is_highlight"] = 0
df_sample.iloc[highlight_indices[0], df_sample.columns.get_loc("is_highlight")] = 1
df_sample.iloc[highlight_indices[1], df_sample.columns.get_loc("is_highlight")] = 1

# Helper to create hovertext for a single row based on hover_col
def _create_hovertext_for_row(row, cols):
    hover_parts = []
    for col in cols:
        if col in row:
            hover_parts.append(f"<b>{col}</b>: {row[col]}")
    return "<br>".join(hover_parts)

hovertext_song_a = _create_hovertext_for_row(df_sample.iloc[highlight_indices[0]], hover_col)
hovertext_song_b = _create_hovertext_for_row(df_sample.iloc[highlight_indices[1]], hover_col)

# -------------------------
# 6. INTERACTIVE 2D & 3D PLOTS FOR ALL MODELS
# -------------------------
for model_name in results.keys():
    print(f"\n{'='*50}")
    print(f"🎨 {model_name.upper()}")
    print('='*50)

    model_embeddings = results[model_name]["embeddings"]
    if len(model_embeddings) != len(df):
        raise ValueError(
            f"Embedding length for {model_name} ({len(model_embeddings)}) does not match df length ({len(df)})."
        )

    X_sample = model_embeddings[idx_sample]

    pca_2d = PCA(n_components=2)
    X_2d = pca_2d.fit_transform(X_sample)
    df_sample["pc1"] = X_2d[:, 0]
    df_sample["pc2"] = X_2d[:, 1]

    pca_3d = PCA(n_components=3)
    X_3d = pca_3d.fit_transform(X_sample)
    df_sample["pc3"] = X_3d[:, 2]

    highlight_2d = df_sample.loc[highlight_indices, ["pc1", "pc2"]]
    highlight_3d = df_sample.loc[highlight_indices, ["pc1", "pc2", "pc3"]]

    distance_2d = np.sqrt(np.sum((highlight_2d.iloc[1] - highlight_2d.iloc[0]) ** 2))
    distance_3d = np.sqrt(np.sum((highlight_3d.iloc[1] - highlight_3d.iloc[0]) ** 2))

    print(f"   Distance (2D PCA): {distance_2d:.4f}")
    print(f"   Distance (3D PCA): {distance_3d:.4f}")

    fig_2d = px.scatter(
        df_sample,
        x="pc1",
        y="pc2",
        hover_data=hover_col,
        title=f"{model_name} — 2D Embedding Map (Color by: {COLOR_BY})"
    )

    fig_2d.update_traces(
        marker=dict(color='gray', size=6, opacity=0.85)
    )

    fig_2d.add_scatter(
        x=df_sample[df_sample["is_highlight"] == 1]["pc1"],
        y=df_sample[df_sample["is_highlight"] == 1]["pc2"],
        mode='markers',
        marker=dict(
            color='red',
            size=10,
            symbol='star',
            line=dict(width=1, color='black')
        ),
        hovertext=[hovertext_song_a, hovertext_song_b],
        hoverinfo='text',
        name='Highlighted Songs',
        showlegend=True
    )

    fig_2d.add_shape(
        type="line",
        x0=highlight_2d.iloc[0, 0],
        y0=highlight_2d.iloc[0, 1],
        x1=highlight_2d.iloc[1, 0],
        y1=highlight_2d.iloc[1, 1],
        line=dict(color='blue', width=2, dash='dash')
    )

    fig_2d.update_layout(
        height=600,
        width=800,
        template='plotly_white',
        hovermode='closest'
    )
    fig_2d.show()

    fig_3d = px.scatter_3d(
        df_sample,
        x="pc1",
        y="pc2",
        z="pc3",
        hover_data=hover_col,
        title=f"{model_name} — 3D Embedding Map (Color by: {COLOR_BY})"
    )

    fig_3d.update_traces(
        marker=dict(color='gray', size=4, opacity=0.9)
    )

    fig_3d.add_trace(go.Scatter3d(
        x=df_sample[df_sample["is_highlight"] == 1]["pc1"],
        y=df_sample[df_sample["is_highlight"] == 1]["pc2"],
        z=df_sample[df_sample["is_highlight"] == 1]["pc3"],
        mode='markers',
        marker=dict(
            color='red',
            size=6,
            symbol='diamond',
            line=dict(width=1, color='black')
        ),
        hovertext=[hovertext_song_a, hovertext_song_b],
        hoverinfo='text',
        name='Highlighted Songs',
        showlegend=True
    ))

    fig_3d.add_trace(go.Scatter3d(
        x=[highlight_3d.iloc[0, 0], highlight_3d.iloc[1, 0]],
        y=[highlight_3d.iloc[0, 1], highlight_3d.iloc[1, 1]],
        z=[highlight_3d.iloc[0, 2], highlight_3d.iloc[1, 2]],
        mode='lines',
        line=dict(color='blue', width=2, dash='dash'),
        name=f"Distance: {distance_3d:.2f}"
    ))

    fig_3d.update_layout(
        height=600,
        width=800,
        template='plotly_white',
        hovermode='closest'
    )
    fig_3d.show()

# -------------------------
# 7. SAFETY CHECK MESSAGES
# -------------------------
if COLOR_BY is not None and COLOR_BY not in df_sample.columns:
    print(f"⚠️ WARNING: '{COLOR_BY}' is not a column in this dataset.")
    print("Available columns:", df_sample.columns.tolist())

# -------------------------
# 8. SUMMARY
# -------------------------
print("\n" + "="*50)
print("✅ PROCESSING & VISUALIZATION COMPLETE")
print("="*50)
print(f"   Models processed: {len(results)}")
print(f"   Models loaded from disk: {len(processed_models)}")
print(f"   New models processed: {len(results) - len(processed_models)}")
print(f"   Visualization graphs created: {len(results)} × 2 (2D + 3D) = {len(results) * 2}")
print(f"   Embeddings saved to local folder")
print("="*50)

print("\n📁 Saved embeddings location:")
print(f"   ./embeddings/")
print("="*50)


🎨 ALL-MINILM-L6-V2
   Distance (2D PCA): 0.4208
   Distance (3D PCA): 0.4441



🎨 ALL-MPNET-BASE-V2
   Distance (2D PCA): 0.3108
   Distance (3D PCA): 0.5962



🎨 MULTI-QA-MINILM-L6-COS-V1
   Distance (2D PCA): 0.4855
   Distance (3D PCA): 0.4899



🎨 MIXEDBREAD-AI/MXBAI-EMBED-LARGE-V1
   Distance (2D PCA): 8.5777
   Distance (3D PCA): 8.6865



✅ PROCESSING & VISUALIZATION COMPLETE
   Models processed: 4
   Models loaded from disk: 4
   New models processed: 0
   Visualization graphs created: 4 × 2 (2D + 3D) = 8
   Embeddings saved to local folder

📁 Saved embeddings location:
   ./embeddings/


## 1.5 Content‑based recommendation

We implement the simplest possible model:

> **Given an item A, recommend items whose embedding is most similar to A.**

We use **cosine similarity** between the embedding of the query item and all others.


# 🎧 Interactive Persona-Based Music Recommendation

This section implements a more advanced interactive recommender specifically for the `music` project. It allows you to:

1.  **Listen to a sample of songs** and provide explicit ratings.
2.  Define a **persona** with textual traits.
3.  Adjust **preference sliders** for audio features like valence, energy, and tempo, as well as mainstream appeal and the weight of your rating history.
4.  Get **real-time recommendations** that combine all these factors.


In [14]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Initialize rated_df globally to store persona ratings
# This DataFrame will hold the user's explicit ratings for sample songs
rated_df = pd.DataFrame()

# --- Recommendation Logic ---
def update_recs(target_valence, target_energy, target_tempo, mainstream, history_match):
    global rated_df # Access the global rated_df

    # 1. Calculate base similarity from user's explicit ratings (if any)
    base_user_vec = None
    if not rated_df.empty:
        # Filter df to get embeddings for rated songs and weight by rating
        # Ensure 'id' in df matches 'track_id' in rated_df
        merged_ratings_df = pd.merge(df[['id']], rated_df, left_on='id', right_on='track_id', how='inner')

        if not merged_ratings_df.empty:
            # Get original indices from df for the rated items
            rated_item_df_indices = df[df['id'].isin(merged_ratings_df['id'])].index.tolist()
            rated_embeddings = embeddings[rated_item_df_indices]

            # Get corresponding ratings in the order of rated_embeddings
            ratings_map = merged_ratings_df.set_index('id')['rating'].to_dict()
            ratings_array = np.array([ratings_map[df.iloc[idx]['id']] for idx in rated_item_df_indices]).reshape(-1, 1)

            if ratings_array.sum() > 0:
                base_user_vec = (rated_embeddings * ratings_array).sum(axis=0) / ratings_array.sum()
                base_user_vec = base_user_vec.reshape(1, -1)

    if base_user_vec is None or rated_df.empty:
        # Fallback if no valid ratings, use the average of all embeddings as a neutral starting point
        base_user_vec = embeddings.mean(axis=0, keepdims=True)

    # Calculate initial similarities based on the user vector
    sims = cosine_similarity(base_user_vec, embeddings)[0]

    # Prepare features for adjustment (use full df features)
    df_features = df[['valence', 'energy', 'tempo', 'track_popularity']].copy()

    # Scale tempo to be between 0 and 1 for consistency with valence/energy
    min_tempo, max_tempo = df_features['tempo'].min(), df_features['tempo'].max()
    df_features['tempo_scaled'] = (df_features['tempo'] - min_tempo) / (max_tempo - min_tempo)

    # Calculate feature-based penalties/boosts
    # Use absolute difference from target, and scale to a penalty
    penalty_v = np.abs(df_features['valence'] - target_valence)
    penalty_e = np.abs(df_features['energy'] - target_energy)
    penalty_t = np.abs(df_features['tempo_scaled'] - (target_tempo - min_tempo) / (max_tempo - min_tempo))

    # Sensitivity (smaller sigma = sharper penalty)
    sigma_v, sigma_e, sigma_t = 0.15, 0.15, 0.15

    feature_based_mod = np.zeros_like(sims)
    feature_based_mod -= (penalty_v / sigma_v) * 0.1 # Valence penalty
    feature_based_mod -= (penalty_e / sigma_e) * 0.1 # Energy penalty
    feature_based_mod -= (penalty_t / sigma_t) * 0.05 # Tempo penalty (slightly less weight)

    # Mainstream adjustment: boost for popular songs if mainstream slider is high
    df_features['popularity_scaled'] = df_features['track_popularity'] / 100.0
    popularity_boost = (df_features['popularity_scaled'] - 0.5) * mainstream * 0.2
    feature_based_mod += popularity_boost

    # Combine embedding similarity and feature-based modification using history_match
    # history_match = 1 -> mostly embedding similarity (from rated items)
    # history_match = 0 -> mostly feature-based modification (from sliders)
    final_scores = (history_match * sims) + ((1 - history_match) * (sims + feature_based_mod)) # Apply feature mod on sims

    # If no ratings were provided, treat history_match as pure feature-based recommendation
    if base_user_vec is embeddings.mean(axis=0, keepdims=True):
         final_scores = sims + feature_based_mod

    # Exclude items that were already rated in the current session
    if not rated_df.empty:
        rated_track_ids_current_session = rated_df['track_id'].tolist()
        # Find indices of these tracks in the main df
        indices_to_exclude = df[df['id'].isin(rated_track_ids_current_session)].index.tolist()
        for idx in indices_to_exclude:
            if idx < len(final_scores):
                final_scores[idx] = -1.0 # Set to a very low score

    k = 8 # Number of recommendations
    top_idx = np.argsort(final_scores)[-k:][::-1]
    recs = df.iloc[top_idx].copy()
    recs["score"] = final_scores[top_idx]

    return display_gallery(recs, title="🎵 Recommended for your Persona")


In [21]:
import numpy as np
import pandas as pd
import requests
import random
import ipywidgets as widgets
from IPython.display import display, HTML, Audio
from sklearn.metrics.pairwise import cosine_similarity

audio_used = []
sample_df = pd.DataFrame()
rated_df = pd.DataFrame()

# --- iTunes API helpers ---
def get_itunes_preview(track_name, artist_name):
    query = f"{track_name} {artist_name}"
    search_url = f"https://itunes.apple.com/search?term={requests.utils.quote(query)}&entity=song&limit=1"
    try:
        resp = requests.get(search_url, timeout=5)
        data = resp.json()
        if data.get('resultCount', 0) > 0:
            return data['results'][0].get('previewUrl')
    except Exception as exc:
        print(f"iTunes request failed: {exc}")
    return None


def test_itunes_api():
    sample_track = ("Watermelon Sugar", "Harry Styles")
    preview = get_itunes_preview(*sample_track)
    if preview:
        display(HTML(f"<p style='color: green;'>✔ iTunes API test passed for sample track '{sample_track[0]}' by {sample_track[1]}.</p>"))
    else:
        display(HTML(f"<p style='color: red;'>⚠ iTunes API test failed for sample track '{sample_track[0]}' by {sample_track[1]}.</p><p>Check network connectivity or iTunes API availability.</p>"))
    return preview


def display_itunes_player(preview_url):
    if preview_url:
        safe_url = preview_url.replace('"', '%22')
        display(HTML(
            f"<audio controls preload='metadata' style='width: 100%; max-width: 550px; margin-bottom: 12px;'>"
            f"<source src=\"{safe_url}\" type='audio/mp4'>"
            "Your browser does not support the audio element."
            "</audio>"
        ))
    else:
        display(HTML("<p style='font-size: 10px; color: gray;'>No audio preview available</p>"))


def display_gallery(recs, title):
    display(HTML(f"<h3>{title}</h3>"))
    if recs.empty:
        display(HTML("<p>No recommendations found.</p>"))
        return

    for _, row in recs.iterrows():
        preview_url = get_itunes_preview(str(row['track_name']), str(row['track_artist']))
        display(HTML(f"<div style='margin-bottom: 6px;'><b>{row['track_name']}</b> - {row['track_artist']} (Score: {row['score']:.3f})</div>"))
        display_itunes_player(preview_url)

# ============================================================================================
# 1) PERSONA INFO
# ============================================================================================
display(HTML("<h2>🎭 Persona Music Preference Simulator</h2>"))
persona_name = widgets.Text(value='', placeholder='e.g., Calm Sofia', description='Persona:', layout=widgets.Layout(width='60%'))
persona_traits = widgets.Textarea(value='', placeholder='Traits...', description='Traits:', layout=widgets.Layout(width='60%', height='70px'))
display(persona_name, persona_traits)

# ============================================================================================
# 2) ITUNES BRIDGE WITH REPLACEMENT LOOP
# ============================================================================================
display(HTML("<h3 style='color: #1DB954;'>🎧 Step 1 — Listen & Rate</h3>"))
test_itunes_api()

def sample_and_display_tracks():
    global sample_df, audio_used, rating_widgets
    final_samples = []
    final_audio = []
    target_count = 10
    picked_ids = set()
    unique_genres = df['playlist_genre'].unique()

    while len(final_samples) < target_count:
        if len(final_samples) < len(unique_genres):
            current_genre = unique_genres[len(final_samples)]
        else:
            current_genre = random.choice(unique_genres)

        genre_pool = df[(df['playlist_genre'] == current_genre) & (~df['id'].isin(picked_ids))]
        if genre_pool.empty:
            continue

        candidate = genre_pool.sample(1).iloc[0]
        preview = get_itunes_preview(str(candidate['track_name']), str(candidate['track_artist']))
        if preview:
            final_samples.append(candidate)
            final_audio.append(preview)
            picked_ids.add(candidate['id'])

    sample_df = pd.DataFrame(final_samples).reset_index(drop=True)
    audio_used = final_audio

    rating_widgets = []
    tracks_elements = []
    for idx, row in sample_df.iterrows():
        preview_url = audio_used[idx]
        safe_url = preview_url.replace('"', '%22')
        tracks_elements.append(HTML(f"<div style='margin-top:15px;'><b>{row['track_name']}</b> — {row['track_artist']}</div>"))
        tracks_elements.append(HTML(
            f"<audio controls preload='metadata' style='width: 100%; max-width: 550px; margin-bottom: 12px;'>"
            f"<source src=\"{safe_url}\" type='audio/mp4'>"
            "Your browser does not support the audio element."
            "</audio>"
        ))
        slider = widgets.IntSlider(value=3, min=1, max=5, step=1, description="Rating:")
        rating_widgets.append(slider)
        tracks_elements.append(slider)
    display(*tracks_elements)

sample_and_display_tracks()

save_button = widgets.Button(description="💾 Save Persona Ratings", button_style="success")
ratings_output = widgets.Output()

def save_persona_ratings(_):
    global rated_df
    persona_ratings = []
    for i, widget in enumerate(rating_widgets):
        row = sample_df.iloc[i]
        persona_ratings.append({
            "persona": persona_name.value,
            "track_id": row["id"],
            "rating": widget.value
        })
    new_ratings = pd.DataFrame(persona_ratings)
    rated_df = pd.concat([rated_df, new_ratings]).drop_duplicates(subset=["track_id"], keep="last")
    with ratings_output:
        ratings_output.clear_output()
        print("✔ Ratings Saved!")

save_button.on_click(save_persona_ratings)
display(save_button, ratings_output)

Text(value='', description='Persona:', layout=Layout(width='60%'), placeholder='e.g., Calm Sofia')

Textarea(value='', description='Traits:', layout=Layout(height='70px', width='60%'), placeholder='Traits...')

IntSlider(value=3, description='Rating:', max=5, min=1)

IntSlider(value=3, description='Rating:', max=5, min=1)

IntSlider(value=3, description='Rating:', max=5, min=1)

...

IntSlider(value=3, description='Rating:', max=5, min=1)

IntSlider(value=3, description='Rating:', max=5, min=1)

IntSlider(value=3, description='Rating:', max=5, min=1)

Button(button_style='success', description='💾 Save Persona Ratings', style=ButtonStyle())

Output()

In [13]:
# ============================================================================================
# 3) RECOMMENDATION UI
# ============================================================================================
display(HTML("<h3 style='margin-top:30px;'>🎯 Step 2 — Recommend with All Embedding Models</h3>"))
valence_s = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, description="Valence")
energy_s = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, description="Energy")
tempo_s = widgets.IntSlider(value=120, min=60, max=200, description="Tempo")
pop_s = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, description="Mainstream")
sim_s = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, description="History Match")


def run_recs(target_v, target_e, target_t, mainstream, history_m):
    global rated_df

    if rated_df.empty:
        display(HTML("<p style='color: orange;'>No persona ratings have been saved yet. Recommendations will use a neutral persona vector.</p>"))

    for model_name in results.keys():
        model_embeddings = results[model_name]["embeddings"]

        if not rated_df.empty:
            m = df[df['id'].isin(rated_df['track_id'])]
            r_map = rated_df.set_index('track_id')['rating'].to_dict()
            r_vals = np.array([r_map[idx] for idx in m['id']]).reshape(-1, 1)
            u_vec = (model_embeddings[m.index] * r_vals).sum(axis=0) / (r_vals.sum() + 1e-9)
            u_vec = u_vec.reshape(1, -1)
        else:
            u_vec = model_embeddings.mean(axis=0, keepdims=True)

        sims = cosine_similarity(u_vec, model_embeddings)[0]
        val_arr = df['valence'].fillna(0.5).values
        en_arr = df['energy'].fillna(0.5).values
        p_v = np.abs(val_arr - target_v)
        p_e = np.abs(en_arr - target_e)
        final_scores = (history_m * sims) + ((1 - history_m) * (sims - (p_v * 0.1) - (p_e * 0.1)))

        if not rated_df.empty:
            idx_to_exclude = df[df['id'].isin(rated_df['track_id'])].index.tolist()
            final_scores[idx_to_exclude] = -1.0

        top = np.argsort(final_scores)[-5:][::-1]
        res = df.iloc[top].copy()
        res['score'] = final_scores[top]
        display_gallery(res, f"🎵 Recommended for you — {model_name}")

out = widgets.interactive_output(run_recs, {"target_v": valence_s, "target_e": energy_s, "target_t": tempo_s, "mainstream": pop_s, "history_m": sim_s})
display(widgets.VBox([valence_s, energy_s, tempo_s, pop_s, sim_s]), out)

Output()

### 💾 Persona & Preference Manager
This tool allows you to save, load, and manage multiple user profiles. Since it stores song IDs and slider values, it is **model-agnostic**: you can swap the underlying embedding model and immediately see how recommendations change for the same persona.

In [11]:
import pickle
import os

# Global storage for personas
# Structure: { name: { 'traits': str, 'ratings': df, 'sliders': dict } }
if 'persona_vault' not in globals():
    persona_vault = {}

def save_persona_to_vault(name, traits, ratings_df, sliders):
    if not name:
        print("❌ Please enter a persona name.")
        return
    persona_vault[name] = {
        'traits': traits,
        'ratings': ratings_df.copy(),
        'sliders': sliders
    }
    print(f"✅ Persona '{name}' saved to vault!")

def delete_persona(name):
    if name in persona_vault:
        del persona_vault[name]
        print(f"🗑️ Persona '{name}' removed.")

# UI for Management
manage_output = widgets.Output()

# Inputs
persona_select = widgets.Dropdown(options=list(persona_vault.keys()), description='Persona:')
btn_load = widgets.Button(description="📂 Load Persona", button_style='info')
btn_delete = widgets.Button(description="🗑️ Delete Persona", button_style='danger')
btn_save_current = widgets.Button(description="💾 Save Current as New", button_style='success')

def refresh_dropdown():
    persona_select.options = list(persona_vault.keys())

def on_save_clicked(_):
    current_sliders = {
        'v': valence_s.value, 'e': energy_s.value, 't': tempo_s.value,
        'm': pop_s.value, 'h': sim_s.value
    }
    save_persona_to_vault(persona_name.value, persona_traits.value, rated_df, current_sliders)
    refresh_dropdown()

def on_load_clicked(_):
    name = persona_select.value
    if name in persona_vault:
        p = persona_vault[name]
        global rated_df
        persona_name.value = name
        persona_traits.value = p['traits']
        rated_df = p['ratings']
        # Update sliders
        valence_s.value = p['sliders']['v']
        energy_s.value = p['sliders']['e']
        tempo_s.value = p['sliders']['t']
        pop_s.value = p['sliders']['m']
        sim_s.value = p['sliders']['h']
        with manage_output:
            manage_output.clear_output()
            print(f"✨ Loaded '{name}'. Recommendations will update automatically.")

def on_delete_clicked(_):
    delete_persona(persona_select.value)
    refresh_dropdown()

btn_save_current.on_click(on_save_clicked)
btn_load.on_click(on_load_clicked)
btn_delete.on_click(on_delete_clicked)

display(HTML("<h3>🗄️ Persona Vault</h3>"))
display(widgets.HBox([persona_select, btn_load, btn_delete]))
display(btn_save_current, manage_output)

Button(button_style='success', description='💾 Save Current as New', style=ButtonStyle())

Output()

---
#  Day 2 — User Profiles & Personalized Recommendation

### Idea

Instead of recommending items **similar to a single item**, we want to recommend items **for a user**.

A simple and surprisingly effective trick:

> Represent the user as the **mean of embeddings of items they liked**.


## 2.1 Build a user profile

For three projects we create a **synthetic user**.  
For movies we can **optionally use real MovieLens ratings** if they exist in this subset.


In [ ]:
def create_synthetic_user_history(n_likes: int = 8) -> np.ndarray:
    """Randomly pick a set of items as a fake user history."""
    n_likes = min(n_likes, len(df))
    return np.random.choice(len(df), size=n_likes, replace=False)


if PROJECT == "movies" and ratings is not None:
    liked = ratings[ratings["rating"] >= 4.0]
    # Count how many liked movies per user
    counts = liked["userId"].value_counts()
    candidate_users = counts[counts >= 15].index.tolist()
    if candidate_users:
        user_id = random.choice(candidate_users)
        user_ratings = liked[liked["userId"] == user_id]
        movie_ids = set(user_ratings["movieId"].astype(str))
        # map movie ids to indices in df
        id_to_idx = {mid: idx for idx, mid in enumerate(df["id"]) if mid in movie_ids}
        user_indices = np.array(list(id_to_idx.values()))
        if len(user_indices) < 5:  # fallback if overlap is small
            user_indices = create_synthetic_user_history(8)
            user_id = None
    else:
        user_indices = create_synthetic_user_history(8)
        user_id = None
else:
    user_indices = create_synthetic_user_history(8)
    user_id = None

print("User liked item indices:", user_indices.tolist())
if PROJECT == "movies" and user_id is not None:
    print("Real MovieLens user id:", user_id)

display(df.iloc[user_indices].head(len(user_indices)))


## 2.2 From liked items to user embedding

We compute the user embedding as the mean of the embeddings of the liked items:

$$
\mathbf{u} = \frac{1}{|L|} \sum_{i \in L} \mathbf{e}_i
$$

where:

- $\mathbf{e}_i$ is the embedding vector of the liked item $i$  
- $L$ is the set of items liked by the user  
- $|L|$ is the number of liked items  

This means that the **user is represented as the centroid of the embeddings of the items they liked** in the latent space.


In [ ]:
user_vec = embeddings[user_indices].mean(axis=0, keepdims=True)
user_sims = cosine_similarity(user_vec, embeddings)[0]
user_sims[user_indices] = -1.0  # do not recommend already liked items

def recommend_for_user(user_sims, k: int = 10) -> pd.DataFrame:
    top_idx = np.argsort(user_sims)[-k:][::-1]
    recs = df.iloc[top_idx].copy()
    recs["score"] = user_sims[top_idx]
    return recs


personal_recs = recommend_for_user(user_sims, k=10)
display(personal_recs.head(10))

display_gallery(personal_recs.head(8), title="Personalized recommendations (user embedding)")


---
# 📅 Day 3 — Hybrid & Constraint‑Aware Recommendation + Interactive Filters

So far we used **only content similarity**.  

In practice, recommenders respect constraints such as:

- budget, preparation time, difficulty, topic/genre, etc.

We now combine:

> **score = similarity − penalty(constraints violation)**.


## 3.1 Extract simple metadata

We define a small helper per project to expose useful attributes.


In [ ]:
def get_metadata(row):
    if PROJECT == "fashion":
        price_str = str(row.get("final_price", np.nan))
        if price_str.startswith('"$'): # Handle cases like "$34.99"
            price_str = price_str.replace('"', '').replace('$', '')
        elif price_str.startswith('$'): # Handle cases like $34.99
            price_str = price_str.replace('$', '')

        try:
            price = float(price_str)
        except ValueError:
            price = np.nan

        return {
            "price": price,
            "color": str(row.get("color", "")),
            "category": str(row.get("category_tree", "")),
        }
    elif PROJECT == "recipes":
        return {
            "complexity": len(str(row.get("ingredients", ""))),
        }
    elif PROJECT == "scholar":
        return {
            "label": str(row.get("label", "")),
        }
    elif PROJECT == "movies":
        return {
            "genres": str(row.get("genres", "")),
        }
    else:
        return {}

## 3.2 Hybrid Recommendation: Combining Similarity and Constraints

Up to now, our recommender worked purely by **semantic similarity**:

- We built a **user embedding** from the items the user liked.
- We computed the **cosine similarity** between this user embedding and every item.
- We recommended the items with the highest similarity scores.

However, **real recommender systems never rely on similarity alone**.

In practice, recommendations are also governed by:
- **Budget constraints** (for shopping),
- **Difficulty or preparation time** (for recipes),
- **Ethical, safety, or policy constraints** (for health, news, etc.),
- **User-specified filters** (genre, color, category, etc.).

This is where **hybrid recommendation** comes in.

---

### 🔹 Core Idea of This Hybrid Model

We start with a **similarity score** for every item:

$$
s_i = \text{cosine}(\mathbf{u}, \mathbf{e}_i)
$$

Then we **modify this score with penalties** whenever an item violates a user constraint:

$$
\tilde{s}_i = s_i - \text{penalty}_i
$$

So the final ranking is based on:

- ✅ How well the item matches the user's taste (similarity)
- ❌ How badly it violates constraints (penalties)

---

### 🔹 What This Function Does Step by Step

For **each item in the dataset**, the function:

1. Starts from the **user–item similarity score**.
2. Looks up the item's **metadata** (price, ingredients, genre, etc.).
3. Applies **penalties** if:
   - The fashion item is **above the user's budget**.
   - The recipe is **too complex**.
4. Explicitly **removes items the user already rated**.
5. Sorts all remaining items by their **final hybrid score**.
6. Returns the **top-k recommended items**.

---

### 🔹 Why This Is a True Hybrid Recommender

This model combines:

- **Content-based personalization** (via embeddings),
- **Rule-based filtering** (via constraints),
- **Context-awareness** (budget, complexity, etc.).

This is exactly the **same principle used in industrial recommender systems**, just in a simplified, transparent form that you can inspect and modify.


In [ ]:
def hybrid_recommend_for_user(user_sims,
                              budget=None,
                              max_complexity=None,
                              k: int = 10) -> pd.DataFrame:
    """
    Hybrid recommender that combines:
    - similarity-based personalization (from user_sims)
    - rule-based constraints (budget, complexity, etc.)

    Parameters
    ----------
    user_sims : np.ndarray
        Similarity score between the user embedding and all item embeddings.
    budget : float or None
        Maximum allowed price (used for fashion).
    max_complexity : int or None
        Maximum allowed recipe complexity (used for recipes).
    k : int
        Number of items to recommend.

    Returns
    -------
    pd.DataFrame
        Top-k recommended items ranked by hybrid score.
    """

    # 1. Start from the pure similarity scores
    scores = user_sims.copy()

    # 2. Loop over all items and apply penalties
    for i in range(len(df)):

        # Extract structured metadata for this item
        meta = get_metadata(df.iloc[i])

        # -------- FASHION CONSTRAINT: BUDGET --------
        if PROJECT == "fashion" and budget is not None:
            price = meta.get("price", np.nan)

            # If the item is overpriced, reduce its score
            if not np.isnan(price) and price > budget:
                scores[i] -= 0.4   # penalty for violating the budget constraint

        # -------- RECIPE CONSTRAINT: COMPLEXITY --------
        if PROJECT == "recipes" and max_complexity is not None:
            comp = meta.get("complexity", 0)

            # If the recipe is too complex, reduce its score
            if comp > max_complexity:
                scores[i] -= 0.3   # penalty for being too complex

        # (you can add more constraints here:
        #  - genre constraints for movies
        #  - category constraints for fashion
        #  - topic constraints for scholarly papers)

    # 3. Do not recommend items the user already rated / liked
    scores[user_indices] = -1.0

    # 4. Rank all items by their final hybrid score
    top_idx = np.argsort(scores)[-k:][::-1]

    # 5. Return the corresponding items
    recs = df.iloc[top_idx].copy()
    recs["score"] = scores[top_idx]

    return recs


In [ ]:
if PROJECT == "fashion":
    hybrid_recs = hybrid_recommend_for_user(
        user_sims,
        budget=30.0,   # user does not want to spend more than 30
        k=10
    )

elif PROJECT == "recipes":
    hybrid_recs = hybrid_recommend_for_user(
        user_sims,
        max_complexity=300,   # avoid long or complicated recipes
        k=10
    )

else:
    hybrid_recs = hybrid_recommend_for_user(user_sims, k=10)

display(hybrid_recs.head(10))
display_gallery(
    hybrid_recs.head(8),
    title="Hybrid recommendations (similarity + constraints)"
)


## 3.3 Interactive filtering (sliders & dropdowns)

We now expose simple **UI controls** on top of our list, so users can:

- adjust max price,  
- limit complexity,  
- filter by genre or label, etc.


In [ ]:
def interactive_filter(recs: pd.DataFrame):
    if PROJECT == "fashion" and "final_price" in recs.columns:
        prices = recs["final_price"].dropna()
        if prices.empty:
            display(recs.head(10))
            return
        min_p, max_p = float(prices.min()), float(prices.max())
        colors = ["(all)"] + sorted(recs["color"].dropna().unique().tolist())

        @interact(max_price=widgets.FloatSlider(value=max_p,
                                                min=min_p, max=max_p,
                                                step=max((max_p - min_p)/20, 1.0)),
                  color=widgets.Dropdown(options=colors))
        def _view(max_price, color):
            subset = recs[recs["final_price"] <= max_price]
            if color != "(all)":
                subset = subset[subset["color"] == color]
            display_gallery(subset.head(12), title=f"Filtered by price<= {max_price:.1f}, color={color}")

    elif PROJECT == "recipes":
        # filter by ingredient string length as complexity
        comps = recs["ingredients"].apply(lambda x: len(str(x)))
        min_c, max_c = float(comps.min()), float(comps.max())

        @interact(max_complexity=widgets.FloatSlider(value=max_c,
                                                     min=min_c, max=max_c,
                                                     step=max((max_c - min_c)/20, 10.0)))
        def _view(max_complexity):
            subset = recs[recs["ingredients"].apply(lambda x: len(str(x))) <= max_complexity]
            display_gallery(subset.head(12), title=f"Recipes complexity <= {max_complexity:.0f}")

    elif PROJECT == "movies":
        # simple genre filter
        all_genres = set()
        for g in recs["genres"].dropna():
            for part in str(g).split("|"):
                all_genres.add(part)
        opts = ["(all)"] + sorted(all_genres)

        @interact(genre=widgets.Dropdown(options=opts))
        def _view(genre):
            subset = recs
            if genre != "(all)":
                subset = subset[subset["genres"].str.contains(genre, na=False)]
            display(subset.head(12))

    elif PROJECT == "scholar":
        labels = ["(all)"] + sorted(recs["label"].dropna().unique().tolist())

        @interact(label=widgets.Dropdown(options=labels))
        def _view(label):
            subset = recs
            if label != "(all)":
                subset = subset[subset["label"] == label]
            display(subset.head(12))
    else:
        display(recs.head(10))


interactive_filter(hybrid_recs)


## 3.4 Playing with different embedding models

If you want to **tune recommendation quality**, you can:

1. Change `EMBEDDING_MODEL` at the top (e.g., `"all-mpnet-base-v2"`).  
2. Re‑run the embedding cell and see how neighbours and recommendations change.

Larger models are usually **slower** but capture more nuanced semantics.


---
# 📅 Day 4 — Agentic Recommender (SimpleAgent)

We now reframe the recommender as a simple **agent**:

1. Parse the user's natural language request.  
2. Translate it into **preferences and constraints**.  
3. Call our hybrid recommender as a **tool**.  
4. Return a ranked list + a short explanation.


In [ ]:
class SimpleAgent:
    def __init__(self, project: str):
        self.project = project

    def parse_request(self, text: str) -> dict:
        t = text.lower()
        prefs = {}

        if self.project == "fashion":
            if "dress" in t:
                prefs["keyword"] = "dress"
            if "black" in t:
                prefs["preferred_color"] = "black"
            if any(w in t for w in ["cheap", "budget", "affordable"]):
                prefs["budget"] = 30.0

        elif self.project == "recipes":
            if "pasta" in t:
                prefs["keyword"] = "pasta"
            if any(w in t for w in ["quick", "fast", "easy"]):
                prefs["max_complexity"] = 250

        elif self.project == "scholar":
            if "recommender" in t:
                prefs["keyword"] = "recommender"
            if any(w in t for w in ["survey", "overview"]):
                prefs["want_overview"] = True

        elif self.project == "movies":
            if "sci-fi" in t or "science fiction" in t:
                prefs["genre"] = "Sci-Fi"
            if "comedy" in t:
                prefs["genre"] = "Comedy"

        return prefs

    def build_query_vector(self, prefs: dict):
        # If we have a keyword, re-encode it as a query; otherwise reuse user_vec
        keyword = prefs.get("keyword", "").strip()
        if keyword:
            return text_model.encode([keyword])
        return user_vec  # fallback to user profile

    def call_recommender(self, q_vec, prefs: dict, k: int = 10) -> pd.DataFrame:
        sims = cosine_similarity(q_vec, embeddings)[0]

        # small genre boost for movies
        if self.project == "movies" and "genre" in prefs:
            g = prefs["genre"]
            boost = np.array([0.2 if g in str(gg) else 0.0 for gg in df["genres"]])
            sims = sims + boost

        recs = hybrid_recommend_for_user(
            sims,
            budget=prefs.get("budget"),
            max_complexity=prefs.get("max_complexity"),
            k=k
        )
        return recs

    def explain(self, prefs: dict) -> str:
        if self.project == "fashion":
            parts = ["I searched for fashion items"]
            if "keyword" in prefs:
                parts.append(f"related to '{prefs['keyword']}'")
            if "preferred_color" in prefs:
                parts.append(f"with color preference '{prefs['preferred_color']}'")
            if "budget" in prefs:
                parts.append(f"while trying to keep price below {prefs['budget']:.0f}")
            return " and ".join(parts) + "."

        if self.project == "recipes":
            parts = ["I looked for recipes"]
            if "keyword" in prefs:
                parts.append(f"containing or similar to '{prefs['keyword']}'")
            if "max_complexity" in prefs:
                parts.append("that are not too complex to prepare")
            return " and ".join(parts) + "."

        if self.project == "scholar":
            parts = ["I retrieved papers"]
            if "keyword" in prefs:
                parts.append(f"about '{prefs['keyword']}'")
            if prefs.get("want_overview"):
                parts.append("and tried to include some overview/survey-style works")
            return " and ".join(parts) + "."

        if self.project == "movies":
            parts = ["I recommended movies"]
            if "genre" in prefs:
                parts.append(f"with genre '{prefs['genre']}'")
            return " and ".join(parts) + "."

        return "I used similarity and simple constraints to select these items."

    def recommend(self, user_text: str, k: int = 6):
        prefs = self.parse_request(user_text)
        q_vec = self.build_query_vector(prefs)
        recs = self.call_recommender(q_vec, prefs, k=k)
        explanation = self.explain(prefs)
        return prefs, recs, explanation


agent = SimpleAgent(PROJECT)


In [ ]:
example_queries = {
    "fashion": "I need a black dress for a party but on a budget.",
    "recipes": "Give me a quick pasta recipe for tonight.",
    "scholar": "I want an overview of recommender systems in healthcare.",
    "movies": "Recommend me some sci-fi movies for a relaxed evening."
}

q = example_queries.get(PROJECT, "Just recommend something.")
prefs, agent_recs, explanation = agent.recommend(q, k=8)

print("User query:", q)
print("Parsed preferences:", prefs)
print("Explanation:", explanation)
display_gallery(agent_recs.head(8), title="Agent-based recommendations")


---

---
# 📅 Day 5 — Critic Agent & Evaluation

Today we:

- Add a simple **CriticAgent** that checks for obvious issues (e.g., low diversity).  
- Compute a basic **diversity metric**.  
- For movies, run a small **offline evaluation** (precision@k).  
- Discuss **offline vs online vs user‑study evaluation** and give a **user study sketch**.


## 5.1 Critic agent

The critic looks at the list and points out basic issues, then optionally modifies it.


In [ ]:
class CriticAgent:
    def __init__(self, project: str):
        self.project = project

    def critique(self, recs: pd.DataFrame) -> dict:
        issues = []
        suggestions = []

        if self.project == "fashion":
            if "final_price" in recs.columns and recs["final_price"].notna().sum() > 0:
                if recs["final_price"].mean() > 50:
                    issues.append("Average price seems high.")
                    suggestions.append("Add some cheaper items for budget‑sensitive users.")
            if "color" in recs.columns and recs["color"].nunique() <= 1:
                issues.append("Low color diversity.")
                suggestions.append("Include items with more varied colors.")

        if self.project == "recipes":
            titles = recs["title"].str.lower().fillna("")
            if titles.str.contains("chicken").sum() >= 3:
                issues.append("Too many chicken recipes.")
                suggestions.append("Add variety in main ingredients (vegetarian, fish, etc.).")

        if self.project == "scholar":
            if "label" in recs.columns:
                frac_ai = (recs["label"] == "ai").mean()
                if frac_ai > 0.8:
                    issues.append("Most abstracts look AI‑generated.")
                    suggestions.append("Include more human‑like papers for balance.")

        if self.project == "movies":
            if "genres" in recs.columns:
                all_genres = "|".join(recs["genres"].fillna(""))
                if all_genres.count("Comedy") >= 4:
                    issues.append("Too many comedy movies.")
                    suggestions.append("Increase genre diversity.")

        return {"issues": issues, "suggestions": suggestions}

    def revise(self, recs: pd.DataFrame) -> pd.DataFrame:
        review = self.critique(recs)
        revised = recs.copy()

        if self.project == "recipes":
            mask = ~revised["title"].str.lower().str.contains("chicken")
            if mask.sum() >= 3:
                revised = revised[mask]

        if self.project == "movies":
            if "genres" in revised.columns:
                mask = ~revised["genres"].str.contains("Comedy", na=False)
                if mask.sum() >= 3:
                    revised = revised[mask]

        return revised.head(5), review


critic = CriticAgent(PROJECT)


In [ ]:
q = example_queries.get(PROJECT, "Just recommend something.")
prefs, agent_recs, explanation = agent.recommend(q, k=10)
revised_recs, review = critic.revise(agent_recs)

print("User query:", q)
print("Agent explanation:", explanation)
print("Critic issues:", review["issues"])
print("Critic suggestions:", review["suggestions"])

display_gallery(revised_recs, title="Final recommendations after critic")


## 5.2 Diversity Metric (Intra-List Diversity)

Accuracy alone is **not sufficient** to evaluate a recommender system.  
If all recommended items are almost identical, the list may be accurate but:

- boring,
- repetitive,
- and not serendipitous.

We therefore measure **intra-list diversity**, which captures **how varied the recommendation list is**.

---

### 🔹 Definition

Let:
- $R$ be the recommended list,
- $|R|$ be the number of recommended items,
- $c(i)$ be the categorical attribute of item $i$ (e.g., color, genre, label).

We define diversity as:

$$
\text{diversity}(R) \;=\; \frac{|\{ c(i) \;:\; i \in R \}|}{|R|}
$$

That is:

> **Number of distinct attribute values divided by the list length.**

---

### 🔹 Interpretation

- **diversity = 1.0** → all recommended items are different  
- **diversity = 0.2** → strong repetition  
- Higher diversity → more exploration & serendipity  
- Lower diversity → more exploitation / narrow taste

---

### 🔹 What Attribute Do We Use?

We adapt the attribute depending on the project:

| Project  | Diversity measured on |
|----------|-------------------------|
| Fashion  | `color` or `category_tree` |
| Recipes  | `title` (proxy for unique dishes) |
| Scholar  | `label` (AI vs Human) |
| Movies   | `main_genre` |

This allows us to **compare diversity before and after hybrid filtering or critic revision**.


In [ ]:
# ============================================
# ✅ INTRA-LIST DIVERSITY METRIC (ROBUST)
# ============================================

def get_diversity_column(PROJECT, recs: pd.DataFrame):
    """Select an appropriate column for diversity computation."""
    if PROJECT == "fashion":
        for col in ["color", "category_tree"]:
            if col in recs.columns:
                return col

    elif PROJECT == "recipes":
        if "title" in recs.columns:
            return "title"

    elif PROJECT == "scholar":
        for col in ["label", "type"]:
            if col in recs.columns:
                return col

    elif PROJECT == "movies":
        # ensure main_genre exists
        if "genres" in recs.columns:
            recs["main_genre"] = recs["genres"].astype(str).apply(lambda x: x.split("|")[0])
            return "main_genre"

    return None


def intra_list_diversity(recs: pd.DataFrame, PROJECT: str) -> float:
    """
    Computes intra-list diversity as:
        (#distinct values in a key column) / (list length)
    """
    if recs is None or len(recs) == 0:
        return np.nan

    col = get_diversity_column(PROJECT, recs)

    if col is None or col not in recs.columns:
        print("⚠️ No suitable column found for diversity computation.")
        return np.nan

    n_distinct = recs[col].nunique(dropna=True)
    diversity = n_distinct / len(recs)

    return float(diversity)


# --------------------------------------------
# ✅ EXAMPLE: DIVERSITY BEFORE VS AFTER CRITIC
# --------------------------------------------

d_before = intra_list_diversity(agent_recs, PROJECT)
d_after  = intra_list_diversity(revised_recs, PROJECT)

print(f"Diversity BEFORE critic: {d_before:.3f}")
print(f"Diversity AFTER  critic: {d_after:.3f}")


In [ ]:
# ============================================
# ✅ OFFLINE EVALUATION FOR ALL PROJECTS
# ============================================

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# -----------------------------
# 1. GENERIC PRECISION@K
# -----------------------------
def precision_at_k(relevant_items, recommended_items, k):
    if len(recommended_items) == 0:
        return 0.0
    recommended_items = recommended_items[:k]
    return len(set(recommended_items) & set(relevant_items)) / k


# -----------------------------
# 2. CONTENT-BASED RECOMMENDER (GENERIC)
# -----------------------------
def recommend_from_user_vector(user_vec, embeddings, k=10):
    sims = cosine_similarity(user_vec, embeddings)[0]
    top_idx = np.argsort(sims)[-k:][::-1]
    return top_idx, sims[top_idx]


# ==========================================================
# ✅ MOVIES: TRUE OFFLINE EVALUATION (MovieLens Precision@K)
# ==========================================================
if PROJECT == "movies" and ratings is not None:

    print("✅ Running TRUE offline evaluation on MovieLens...")

    ratings_filtered = ratings[ratings["rating"] >= 4]
    user_ids = ratings_filtered["userId"].unique()

    precisions = []

    for uid in user_ids[:100]:  # evaluate on 100 users for speed

        user_data = ratings_filtered[ratings_filtered["userId"] == uid]
        liked_items = user_data["movieId"].astype(str).tolist()

        if len(liked_items) < 5:
            continue

        # Train/test split
        np.random.shuffle(liked_items)
        train_likes = liked_items[:int(0.7 * len(liked_items))]
        test_likes  = liked_items[int(0.7 * len(liked_items)):]

        # Build user vector from training items
        train_indices = df[df["id"].isin(train_likes)].index.tolist()
        if len(train_indices) == 0:
            continue

        user_vec = embeddings[train_indices].mean(axis=0).reshape(1, -1)

        # Recommend
        recommended_idx, _ = recommend_from_user_vector(user_vec, embeddings, k=10)
        recommended_ids = df.iloc[recommended_idx]["id"].tolist()

        # Precision@10
        p = precision_at_k(test_likes, recommended_ids, k=10)
        precisions.append(p)

    print(f"✅ MovieLens Precision@10: {np.mean(precisions):.4f}")


# ==================================================
# ✅ PROXY OFFLINE EVALUATION (FASHION / RECIPES / SCHOLAR)
# ==================================================
else:
    print("⚠️ No true ground truth available — running PROXY evaluation")

    def create_proxy_ground_truth(df, PROJECT):
        if PROJECT == "fashion":
            return df["product_name"].astype(str).sample(500).index.tolist()
        elif PROJECT == "recipes":
            return df["title"].astype(str).sample(500).index.tolist()
        elif PROJECT == "scholar":
            return df["label"].astype(str).sample(500).index.tolist()
        else:
            return []

    proxy_relevant = create_proxy_ground_truth(df, PROJECT)

    # Build a pseudo-user from 5 random liked items
    seed_idx = np.random.choice(len(df), 5, replace=False)
    user_vec = embeddings[seed_idx].mean(axis=0).reshape(1, -1)

    recommended_idx, _ = recommend_from_user_vector(user_vec, embeddings, k=10)

    precision_proxy = precision_at_k(proxy_relevant, recommended_idx.tolist(), k=10)

    print(f"⚠️ Proxy Precision@10 for {PROJECT}: {precision_proxy:.4f}")


## 5.3 Offline Evaluation of Recommender Systems (All Projects)

Evaluation asks a fundamental question:

> "How good are the recommendations, objectively?"

Recommender systems are typically evaluated in **three complementary ways**:

---

## ① Offline Evaluation (Using Historical Data)

This is done using **logged interaction data** such as:
- ratings,
- clicks,
- listens,
- purchases.

We simulate the real system **without deploying it**.

Typical metrics:
- Precision@K
- Recall@K
- NDCG@K
- Coverage
- Diversity

✅ Fast  
✅ Reproducible  
❌ Limited realism  
❌ Offline bias (users never saw unseen items)

---

## ② Online Evaluation (A/B Testing)

Two or more algorithms are deployed to real users:

- Group A → algorithm 1  
- Group B → algorithm 2  

We compare:
- Click-through rate
- Engagement
- Conversion
- Retention

✅ Gold standard  
❌ Expensive  
❌ Requires real users & deployment

---

## ③ User Studies (Human-Centered Evaluation)

Participants:
- Receive recommendations
- Complete questionnaires:
  - relevance
  - diversity
  - trust
  - explanation quality
  - sense of control
  - emotional impact

✅ Best for:
- health,
- art,
- education,
- explainable AI  
❌ Smaller sample sizes  
❌ Slower

---

# ✅ How Evaluation Applies to Our Four Projects

| Project  | Offline GT available? | Offline Metric Used | User Study Suitable? |
|----------|------------------------|---------------------|-----------------------|
| Movies   | ✅ Yes (MovieLens)     | Precision@K         | ✅ Yes |
| Fashion  | ❌ No                  | Proxy Precision     | ✅ Yes |
| Recipes  | ❌ No                  | Proxy Precision     | ✅ Yes |
| Scholar  | ❌ No                  | Pseudo Label Match  | ✅ Yes |

Only **movies** contains true historical preference labels.  
The others require **proxy relevance assumptions**.

---

# ✅ Precision@K Definition

Let:
- $T$ = set of test items (ground truth)
- $R_k$ = top-$k$ recommended items

Then:

$$
\text{Precision@k} = \frac{|R_k \cap T|}{k}
$$

Interpretation:
- Precision@10 = 0.4 → 4 out of the 10 recommended items were actually relevant.

---

# ✅ Why We Still Evaluate Non-Movie Projects Offline

Even without true labels:
- We can simulate **pseudo ground truth**
- This allows:
  - algorithm comparison,
  - ablation studies,
  - bias diagnosis,
  - architecture testing.

Final validation always requires:
✅ user studies.


## 5.4 How to evaluate recommender systems (conceptual)

In general there are **three main families** of evaluation:

### 1. Offline evaluation (what we approximated above)
- Uses **historical logs** (ratings, clicks).  
- Split data into train / validation / test.  
- Compute metrics like: Precision@k, Recall@k, NDCG, MAP, coverage, diversity.  
- Pros: cheap, safe, fast.  
- Cons: reflects **past** exposure; may be biased by previous recommender.

### 2. Online evaluation (A/B tests)
- Deploy two or more variants (A vs B) to real users.  
- Randomly assign users to conditions.  
- Log click‑through rate, conversion, dwell time, etc.  
- Use statistical tests to see if B is better than A.  
- Powerful but requires **traffic, engineering, and ethics**.

### 3. User studies (controlled experiments / lab or field)
- Recruit participants and give them tasks (e.g., “Find a movie for Friday night”).  
- Log behaviour, but also ask **subjective questions**, e.g.:  
  - Perceived **accuracy** of recommendations  
  - Perceived **diversity** and **serendipity**  
  - Trust, transparency, control, satisfaction  
  - Quality of **explanations** and **controls** (sliders, filters)

In this lab we:

- Implemented a small **offline evaluation** (movies).  
- Sketched a **user‑study interface** below.


## 5.5 Sketch for a small user study (Gradio UI)

You can turn your agent into a simple web app and share it with a few friends.

Participants could be asked to:

- Enter a persona or request (text).  
- Rate the resulting recommendations on:
  - Accuracy (1–5)  
  - Diversity (1–5)  
  - Serendipity (1–5) — “Did you discover anything pleasantly surprising?”  
  - Explanation quality (1–5)  
  - Control / usability of filters (1–5)  
- Optionally provide free‑text comments.

Below is a minimal Gradio sketch. Uncomment and run if you want to try it.


In [ ]:
# ============================================================
# ✅  GRADIO USER STUDY
# ============================================================

!pip install gradio --quiet

import gradio as gr
import pandas as pd
import datetime
import random

# ============================================================
# ✅ 1. DOMAIN CONFIG
# ============================================================

DOMAIN_CONFIG = {
    "fashion": {
        "title": "👜 Fashion Recommender User Study",
        "example": "Minimalist outfit under 30€",
        "header_img": "https://images.unsplash.com/photo-1521335629791-ce4aec67dd47"
    },
    "recipes": {
        "title": "🍽️ Recipe Recommender User Study",
        "example": "Easy vegetarian dinner",
        "header_img": "https://images.unsplash.com/photo-1498837167922-ddd27525d352"
    },
    "scholar": {
        "title": "📚 Research Paper Recommender User Study",
        "example": "AI fairness and bias paper",
        "header_img": "https://images.unsplash.com/photo-1532012197267-da84d127e765"
    },
    "movies": {
        "title": "🎬 Movie Recommender User Study",
        "example": "Smart sci-fi movies like Ex Machina",
        "header_img": "https://images.unsplash.com/photo-1489599849927-2ee91cede3ba"
    }
}

cfg = DOMAIN_CONFIG.get(PROJECT, DOMAIN_CONFIG["movies"])

# ============================================================
# ✅ 2. STATIC REAL IMAGE POOLS (DIRECT FILE LINKS)
# ============================================================

IMAGE_POOL = {
    "recipes": [
        "https://upload.wikimedia.org/wikipedia/commons/0/0c/Spaghetti_al_Pomodoro.JPG",
        "https://upload.wikimedia.org/wikipedia/commons/2/2c/Chicken_alfredo_fettuccine.JPG",
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/Lasagne_-_stonesoup.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/0/0b/Penne_all%27arrabbiata.jpg"
    ],
    "fashion": [
        "https://upload.wikimedia.org/wikipedia/commons/4/4e/Evening_gown_MET_2014.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/0/04/Menswear_suit.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/1/1b/Clothing_fashion_store.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/3/3f/Fashion_model_in_weird_dress.jpg"
    ],
    "movies": [
        "https://upload.wikimedia.org/wikipedia/commons/2/29/Film_projector.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/6/6b/Cinema_audience.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/5/58/Movie_theater_screen.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/7/7a/Film_reel_icon.png"
    ],
    "scholar": [
        "https://upload.wikimedia.org/wikipedia/commons/8/8c/Neural_network.svg",
        "https://upload.wikimedia.org/wikipedia/commons/1/1a/Artificial_intelligence_outline.svg",
        "https://upload.wikimedia.org/wikipedia/commons/3/3c/Scientific_research.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/4/4c/Bookshelf_icon.png"
    ]
}

def get_static_image():
    return random.choice(IMAGE_POOL.get(PROJECT, IMAGE_POOL["movies"]))

# ============================================================
# ✅ 3. BUILD CARD (REAL IMAGE + ITEM INFO)
# ============================================================

def build_card_html(row):

    if PROJECT == "fashion":
        title = str(row.get("product_name", "Fashion Item"))
        meta = f"Color: {row.get('color','N/A')} | Price: {row.get('final_price','N/A')}"

    elif PROJECT == "recipes":
        title = str(row.get("title", "Recipe"))
        meta = "Ingredients: " + str(row.get("ingredients",""))[:80] + "..."

    elif PROJECT == "scholar":
        title = str(row.get("title", "Research Paper"))
        meta = "Label: " + str(row.get("label", "N/A"))

    else:  # movies
        title = str(row.get("title", "Movie"))
        meta = "Genres: " + str(row.get("genres", "N/A"))

    img_url = get_static_image()

    return f"""
    <div style="
        background:white;
        border-radius:16px;
        box-shadow:0 4px 10px rgba(0,0,0,0.25);
        overflow:hidden;
        text-align:center;
        padding-bottom:10px;
    ">
        <img src="{img_url}" style="width:100%;height:220px;object-fit:cover;">
        <div style="padding:8px;">
            <b>{title}</b><br>
            <small>{meta}</small>
        </div>
    </div>
    """

# ============================================================
# ✅ 4. AGENT → GRID OUTPUT
# ============================================================

def gradio_recommend(query):

    try:
        _, recs, explanation = agent.recommend(query, k=6)
    except:
        recs = agent.recommend(query, k=6)
        explanation = "No explanation generated."

    grid_html = "<div style='display:grid;grid-template-columns:repeat(3,1fr);gap:16px;'>"

    for _, row in recs.iterrows():
        grid_html += build_card_html(row)

    grid_html += "</div>"

    return grid_html, explanation

# ============================================================
# ✅ 5. CSV LOGGING
# ============================================================

LOG_FILE = "user_study_results.csv"

def save_user_feedback(query, recs_html,
                       acc, div, ser, expl, ctrl,
                       comment):

    ts = datetime.datetime.now().isoformat()

    row = {
        "timestamp": ts,
        "project": PROJECT,
        "query": query,
        "recommendations": recs_html.replace(",", ";"),
        "accuracy": acc,
        "diversity": div,
        "serendipity": ser,
        "explanation_quality": expl,
        "control_usability": ctrl,
        "comment": comment
    }

    try:
        df_log = pd.read_csv(LOG_FILE)
        df_log = pd.concat([df_log, pd.DataFrame([row])], ignore_index=True)
    except:
        df_log = pd.DataFrame([row])

    df_log.to_csv(LOG_FILE, index=False)

    return "✅ Feedback saved!"

# ============================================================
# ✅ 6. GRADIO UI (HEADER + GRID)
# ============================================================

with gr.Blocks(title=cfg["title"]) as demo:

    # ✅ HEADER — SAME AS BEFORE (WORKING)
    gr.HTML(f"""
    <div style="text-align:center;">
        <img src="{cfg['header_img']}"
             style="width:100%;max-height:320px;
                    object-fit:cover;border-radius:20px;">
        <h1 style="margin-top:16px;">{cfg['title']}</h1>
        <p><i>Example: {cfg['example']}</i></p>
    </div>
    """)

    query_input = gr.Textbox(lines=2, label="Describe what you want")

    recommend_btn = gr.Button("🔍 Generate Recommendations")

    recs_output = gr.HTML(label="Visual Recommendations")
    explanation_output = gr.Textbox(label="System Explanation")

    recommend_btn.click(
        gradio_recommend,
        inputs=query_input,
        outputs=[recs_output, explanation_output]
    )

    gr.Markdown("## ✅ Rate the recommendations")

    with gr.Row():
        acc_slider  = gr.Slider(1, 5, step=1, label="Accuracy")
        div_slider  = gr.Slider(1, 5, step=1, label="Diversity")
        ser_slider  = gr.Slider(1, 5, step=1, label="Serendipity")
        expl_slider = gr.Slider(1, 5, step=1, label="Explanation Quality")
        ctrl_slider = gr.Slider(1, 5, step=1, label="Control & Usability")

    comment_box = gr.Textbox(lines=3, label="Optional feedback")

    submit_btn = gr.Button("📩 Submit Feedback")
    status_output = gr.Textbox(label="Status")

    submit_btn.click(
        save_user_feedback,
        inputs=[
            query_input,
            recs_output,
            acc_slider,
            div_slider,
            ser_slider,
            expl_slider,
            ctrl_slider,
            comment_box
        ],
        outputs=status_output
    )

demo.launch(share=True)


In [ ]:
# ============================================
# ✅ USER STUDY ANALYSIS DASHBOARD
# ============================================

import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
from wordcloud import WordCloud

df_log = pd.read_csv("user_study_results.csv")

metrics = ["accuracy", "diversity", "serendipity", "explanation_quality", "control_usability"]

# -----------------------------
# ✅ BAR PLOT (MEAN SCORES)
# -----------------------------
mean_scores = df_log[metrics].mean()

plt.figure()
mean_scores.plot(kind="bar")
plt.title("Mean User Ratings Across Metrics")
plt.ylabel("Score (1–5)")
plt.show()

# -----------------------------
# ✅ BOX PLOTS (DISTRIBUTIONS)
# -----------------------------
plt.figure()
df_log[metrics].plot(kind="box")
plt.title("Distribution of User Ratings")
plt.show()

# -----------------------------
# ✅ SENTIMENT ANALYSIS
# -----------------------------
def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

df_log["sentiment"] = df_log["comment"].apply(get_sentiment)

plt.figure()
df_log["sentiment"].hist(bins=20)
plt.title("Sentiment Distribution of User Comments")
plt.show()

print("Average Comment Sentiment:", df_log["sentiment"].mean())

# -----------------------------
# ✅ WORD CLOUD (COMMENTS)
# -----------------------------
all_comments = " ".join(df_log["comment"].dropna().astype(str))

wc = WordCloud(width=800, height=400, background_color="white").generate(all_comments)

plt.figure()
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of User Feedback")
plt.show()


In [ ]:
# ============================================================
# ✅ A/B USER STUDY APP (Alg A: Agentic, Alg B: Rating-based)
# ============================================================

!pip install gradio --quiet

import gradio as gr
import pandas as pd
import datetime
import random
import uuid
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# 0. EXPECTED GLOBALS FROM NOTEBOOK:
# - PROJECT  (str: "fashion", "recipes", "scholar", "movies")
# - df       (pandas DataFrame of items)
# - embeddings (numpy array of item embeddings, len(df) rows)
# - agent    (object with .recommend(query, k) for Alg A)
# ------------------------------------------------------------

# ------------------------------------------------------------
# 1. DOMAIN CONFIG (HEADER IMAGE + TITLE)
# ------------------------------------------------------------
DOMAIN_CONFIG = {
    "fashion": {
        "title": "👜 Fashion Recommender User Study (A/B)",
        "example": "Minimalist outfit under 30€",
        "header_img": "https://images.unsplash.com/photo-1521335629791-ce4aec67dd47"
    },
    "recipes": {
        "title": "🍽️ Recipe Recommender User Study (A/B)",
        "example": "Easy vegetarian dinner",
        "header_img": "https://images.unsplash.com/photo-1498837167922-ddd27525d352"
    },
    "scholar": {
        "title": "📚 Research Paper Recommender User Study (A/B)",
        "example": "AI fairness and bias paper",
        "header_img": "https://images.unsplash.com/photo-1532012197267-da84d127e765"
    },
    "movies": {
        "title": "🎬 Movie Recommender User Study (A/B)",
        "example": "Smart sci-fi movies like Ex Machina",
        "header_img": "https://images.unsplash.com/photo-1489599849927-2ee91cede3ba"
    }
}

cfg = DOMAIN_CONFIG.get(PROJECT, DOMAIN_CONFIG["movies"])

# ------------------------------------------------------------
# 2. STATIC REAL IMAGE POOLS (WORKING, NON-BROKEN)
# ------------------------------------------------------------
IMAGE_POOL = {
    "recipes": [
        "https://upload.wikimedia.org/wikipedia/commons/0/0c/Spaghetti_al_Pomodoro.JPG",
        "https://upload.wikimedia.org/wikipedia/commons/2/2c/Chicken_alfredo_fettuccine.JPG",
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/Lasagne_-_stonesoup.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/0/0b/Penne_all%27arrabbiata.jpg"
    ],
    "fashion": [
        "https://upload.wikimedia.org/wikipedia/commons/4/4e/Evening_gown_MET_2014.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/0/04/Menswear_suit.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/1/1b/Clothing_fashion_store.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/3/3f/Fashion_model_in_weird_dress.jpg"
    ],
    "movies": [
        "https://upload.wikimedia.org/wikipedia/commons/2/29/Film_projector.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/6/6b/Cinema_audience.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/5/58/Movie_theater_screen.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/7/7a/Film_reel_icon.png"
    ],
    "scholar": [
        "https://upload.wikimedia.org/wikipedia/commons/8/8c/Neural_network.svg",
        "https://upload.wikimedia.org/wikipedia/commons/1/1a/Artificial_intelligence_outline.svg",
        "https://upload.wikimedia.org/wikipedia/commons/3/3c/Scientific_research.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/4/4c/Bookshelf_icon.png"
    ]
}

def get_static_image():
    return random.choice(IMAGE_POOL.get(PROJECT, IMAGE_POOL["movies"]))

# ------------------------------------------------------------
# 3. VISUAL CARD BUILDER (USED BY BOTH Alg A & B)
# ------------------------------------------------------------
def build_card_html(row):
    """Build a visual recommendation card with image + text."""
    if PROJECT == "fashion":
        title = str(row.get("product_name", "Fashion Item"))
        meta = f"Color: {row.get('color','N/A')} | Price: {row.get('final_price','N/A')}"
    elif PROJECT == "recipes":
        title = str(row.get("title", "Recipe"))
        meta = "Ingredients: " + str(row.get("ingredients",""))[:80] + "..."
    elif PROJECT == "scholar":
        title = str(row.get("title", "Research Paper"))
        meta = "Label: " + str(row.get("label", "N/A"))
    else:  # movies
        title = str(row.get("title", "Movie"))
        meta = "Genres: " + str(row.get("genres", "N/A"))

    img_url = get_static_image()

    return f"""
    <div style="
        background:white;
        border-radius:16px;
        box-shadow:0 4px 10px rgba(0,0,0,0.25);
        overflow:hidden;
        text-align:center;
        padding-bottom:10px;
    ">
        <img src="{img_url}" style="width:100%;height:220px;object-fit:cover;">
        <div style="padding:8px;">
            <b>{title}</b><br>
            <small>{meta}</small>
        </div>
    </div>
    """

def recs_to_grid_html(recs: pd.DataFrame):
    """Convert a DataFrame of items into a visual HTML grid."""
    grid_html = "<div style='display:grid;grid-template-columns:repeat(3,1fr);gap:16px;'>"
    for _, row in recs.iterrows():
        grid_html += build_card_html(row)
    grid_html += "</div>"
    return grid_html

# ------------------------------------------------------------
# 4. ALG A = AGENTIC / PROMPT-BASED RECOMMENDER
# ------------------------------------------------------------
def run_alg_A(query: str):
    """
    Use the agent-based recommender to get recommendations.
    Expected agent.recommend(query, k) → (indices, df_recs, explanation) OR df_recs.
    """
    if not query.strip():
        return "⚠️ Please enter a persona / request first.", ""

    try:
        _, recs, explanation = agent.recommend(query, k=6)
    except:
        recs = agent.recommend(query, k=6)
        explanation = "Agent did not return an explicit explanation."

    grid_html = recs_to_grid_html(recs)
    return grid_html, explanation

# ------------------------------------------------------------
# 5. ALG B = RATING-BASED RECOMMENDER (EXPLICIT FEEDBACK)
# ------------------------------------------------------------

def sample_items_for_B():
    """Sample 5 items from df for rating."""
    idx = random.sample(range(len(df)), 5)
    return idx

def describe_item_for_B(row):
    """Short text description shown next to rating sliders in Alg B."""
    if PROJECT == "fashion":
        return f"{row.get('product_name','Item')} ({row.get('color','?')}, {row.get('final_price','?')})"
    elif PROJECT == "recipes":
        return f"{row.get('title','Recipe')} – ingredients: {str(row.get('ingredients',''))[:60]}..."
    elif PROJECT == "scholar":
        return f"{row.get('title','Paper')} – {row.get('label','')}"
    else:
        return f"{row.get('title','Movie')} – {row.get('genres','')}"

def run_alg_B_get_recs(sample_indices, r1, r2, r3, r4, r5):
    """
    Compute a recommendation list based on the user's ratings
    for the 5 sampled items.
    """
    if sample_indices is None or len(sample_indices) != 5:
        return "⚠️ Please click 'Sample items to rate' first.", ""

    ratings = [r1, r2, r3, r4, r5]
    indices = sample_indices

    # Build user embedding as weighted avg of item embeddings
    item_vecs = embeddings[indices]
    weights = np.array(ratings).reshape(-1, 1)

    if weights.sum() == 0:
        user_vec = item_vecs.mean(axis=0, keepdims=True)
    else:
        user_vec = (item_vecs * weights).sum(axis=0, keepdims=True) / weights.sum()

    sims = cosine_similarity(user_vec, embeddings)[0]

    # Exclude sampled items themselves
    for i in indices:
        sims[i] = -1.0

    top_idx = np.argsort(sims)[-6:][::-1]
    recs = df.iloc[top_idx].copy()
    recs["score"] = sims[top_idx]

    grid_html = recs_to_grid_html(recs)
    explanation = (
        "These recommendations are based on your explicit ratings of a few items "
        "(rating-based Algorithm B)."
    )
    return grid_html, explanation

# ------------------------------------------------------------
# 6. CSV LOGGING WITH PARTICIPANT ID + ALGORITHM LABEL
# ------------------------------------------------------------
LOG_FILE_AB = "user_study_results_ab.csv"

def save_feedback(participant_id,
                  algorithm_label,
                  query_or_prompt,
                  rated_items_str,
                  ratings_str,
                  recs_html,
                  acc, div, ser, expl_q, ctrl, comment):

    ts = datetime.datetime.now().isoformat()

    row = {
        "timestamp": ts,
        "participant_id": participant_id,
        "project": PROJECT,
        "algorithm": algorithm_label,  # "A" or "B"
        "query_or_prompt": query_or_prompt,
        "rated_items": rated_items_str,
        "rated_item_scores": ratings_str,
        "recommendations_html": recs_html.replace(",", ";"),
        "accuracy": acc,
        "diversity": div,
        "serendipity": ser,
        "explanation_quality": expl_q,
        "control_usability": ctrl,
        "comment": comment
    }

    try:
        df_log = pd.read_csv(LOG_FILE_AB)
        df_log = pd.concat([df_log, pd.DataFrame([row])], ignore_index=True)
    except:
        df_log = pd.DataFrame([row])

    df_log.to_csv(LOG_FILE_AB, index=False)
    return "✅ Feedback saved!"

# ------------------------------------------------------------
# 7. PARTICIPANT ID GENERATOR
# ------------------------------------------------------------
def generate_participant_id():
    return "P-" + uuid.uuid4().hex[:8]

# ------------------------------------------------------------
# 8. BUILD GRADIO A/B UI
# ------------------------------------------------------------
with gr.Blocks(title=cfg["title"]) as demo_ab:

    gr.HTML(f"""
    <div style="text-align:center;">
        <img src="{cfg['header_img']}"
             style="width:100%;max-height:320px;
                    object-fit:cover;border-radius:20px;">
        <h1 style="margin-top:16px;">{cfg['title']}</h1>
        <p><i>Example prompt for Alg A: {cfg['example']}</i></p>
    </div>
    """)

    # ---------- Participant ID ----------
    with gr.Row():
        participant_id_state = gr.State()
        pid_box = gr.Textbox(label="Participant ID (anonymous)", interactive=False)
        gen_id_btn = gr.Button("🔑 Generate / Reset ID")

    def init_id(_):
        new_id = generate_participant_id()
        return new_id, new_id

    gen_id_btn.click(
        init_id,
        inputs=[participant_id_state],
        outputs=[pid_box, participant_id_state]
    )

    gr.Markdown("### Choose which algorithm to try (A/B):")

    with gr.Tab("Algorithm A – Agentic (prompt-based)"):

        # ---- Alg A: query + recs ----
        query_A = gr.Textbox(lines=2, label="Describe your persona / mood / constraints")
        btn_A = gr.Button("🔍 Get recommendations (Alg A)")

        recs_A_html = gr.HTML(label="Recommendations (Alg A)")
        expl_A_box = gr.Textbox(label="Agent Explanation")

        btn_A.click(
            run_alg_A,
            inputs=[query_A],
            outputs=[recs_A_html, expl_A_box]
        )

        gr.Markdown("#### Rate Algorithm A")

        with gr.Row():
            acc_A = gr.Slider(1, 5, step=1, label="Accuracy (Alg A)")
            div_A = gr.Slider(1, 5, step=1, label="Diversity (Alg A)")
            ser_A = gr.Slider(1, 5, step=1, label="Serendipity (Alg A)")
            explq_A = gr.Slider(1, 5, step=1, label="Explanation quality (Alg A)")
            ctrl_A = gr.Slider(1, 5, step=1, label="Control & usability (Alg A)")

        comment_A = gr.Textbox(lines=3, label="Optional comments about Alg A")
        submit_A = gr.Button("📩 Submit feedback for Alg A")
        status_A = gr.Textbox(label="Status (Alg A)", interactive=False)

        def submit_feedback_A(pid, query, recs_html, acc, d, s, eq, cu, comment):
            if not pid:
                return "⚠️ Please generate a Participant ID first."
            return save_feedback(
                participant_id=pid,
                algorithm_label="A",
                query_or_prompt=query,
                rated_items_str="",
                ratings_str="",
                recs_html=recs_html,
                acc=acc, div=d, ser=s, expl_q=eq, ctrl=cu, comment=comment
            )

        submit_A.click(
            submit_feedback_A,
            inputs=[
                participant_id_state,
                query_A,
                recs_A_html,
                acc_A, div_A, ser_A, explq_A, ctrl_A,
                comment_A
            ],
            outputs=status_A
        )

    with gr.Tab("Algorithm B – Rating-based"):

        gr.Markdown(
            "In Algorithm B, you first **rate a few items**, then the system recommends similar items."
        )

        # Sample items to rate
        sample_indices_state = gr.State()

        sample_btn = gr.Button("🎲 Sample 5 items to rate")

        # 5 items + 5 rating sliders
        item1_text = gr.Markdown("")
        item2_text = gr.Markdown("")
        item3_text = gr.Markdown("")
        item4_text = gr.Markdown("")
        item5_text = gr.Markdown("")

        r1 = gr.Slider(1, 5, step=1, value=3, label="Rating for item 1")
        r2 = gr.Slider(1, 5, step=1, value=3, label="Rating for item 2")
        r3 = gr.Slider(1, 5, step=1, value=3, label="Rating for item 3")
        r4 = gr.Slider(1, 5, step=1, value=3, label="Rating for item 4")
        r5 = gr.Slider(1, 5, step=1, value=3, label="Rating for item 5")

        def prepare_items_for_B():
            idx = sample_items_for_B()
            rows = [df.iloc[i] for i in idx]
            descs = [describe_item_for_B(r) for r in rows]
            return (
                idx,
                f"**Item 1:** {descs[0]}",
                f"**Item 2:** {descs[1]}",
                f"**Item 3:** {descs[2]}",
                f"**Item 4:** {descs[3]}",
                f"**Item 5:** {descs[4]}",
            )

        sample_btn.click(
            prepare_items_for_B,
            inputs=[],
            outputs=[sample_indices_state, item1_text, item2_text, item3_text, item4_text, item5_text]
        )

        # Recs from Alg B
        btn_B = gr.Button("🔍 Get recommendations (Alg B)")

        recs_B_html = gr.HTML(label="Recommendations (Alg B)")
        expl_B_box = gr.Textbox(label="Explanation (Alg B)")

        def run_B_wrapper(sample_idx, r1, r2, r3, r4, r5):
            return run_alg_B_get_recs(sample_idx, r1, r2, r3, r4, r5)

        btn_B.click(
            run_B_wrapper,
            inputs=[sample_indices_state, r1, r2, r3, r4, r5],
            outputs=[recs_B_html, expl_B_box]
        )

        gr.Markdown("#### Rate Algorithm B")

        with gr.Row():
            acc_B = gr.Slider(1, 5, step=1, label="Accuracy (Alg B)")
            div_B = gr.Slider(1, 5, step=1, label="Diversity (Alg B)")
            ser_B = gr.Slider(1, 5, step=1, label="Serendipity (Alg B)")
            explq_B = gr.Slider(1, 5, step=1, label="Explanation quality (Alg B)")
            ctrl_B = gr.Slider(1, 5, step=1, label="Control & usability (Alg B)")

        comment_B = gr.Textbox(lines=3, label="Optional comments about Alg B")
        submit_B = gr.Button("📩 Submit feedback for Alg B")
        status_B = gr.Textbox(label="Status (Alg B)", interactive=False)

        def submit_feedback_B(pid, sample_idx, r1, r2, r3, r4, r5,
                              recs_html, acc, d, s, eq, cu, comment):
            if not pid:
                return "⚠️ Please generate a Participant ID first."

            if sample_idx is None:
                rated_items_str = ""
                ratings_str = ""
            else:
                rated_items_str = ",".join(str(df.iloc[i].get("id", i)) for i in sample_idx)
                ratings_str = ",".join(str(x) for x in [r1, r2, r3, r4, r5])

            return save_feedback(
                participant_id=pid,
                algorithm_label="B",
                query_or_prompt="(rating-based)",
                rated_items_str=rated_items_str,
                ratings_str=ratings_str,
                recs_html=recs_html,
                acc=acc, div=d, ser=s, expl_q=eq, ctrl=cu, comment=comment
            )

        submit_B.click(
            submit_feedback_B,
            inputs=[
                participant_id_state,
                sample_indices_state,
                r1, r2, r3, r4, r5,
                recs_B_html,
                acc_B, div_B, ser_B, explq_B, ctrl_B,
                comment_B
            ],
            outputs=status_B
        )

demo_ab.launch(share=True)
